# Cell 1: Environment Setup

In [1]:
import os
import math
import copy
import time
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, balanced_accuracy_score, roc_auc_score, confusion_matrix
)


# ==============================================================================
# v11 LACA NUM_HEADS SWEEP — every choice below is FIXED (v10 settings: beta=1.0,
# mlp lag-bias g(Δτ)) except the number of LACA attention heads, which is the
# only variable. SEED is the single source for all seeding in this notebook.
# ==============================================================================
SEED = 42
LACA_BETA = 1.0              # LACA temporal_bias_scale — FIXED (v9 sweep winner)
LAG_BIAS_FORM = "mlp"        # LACA lag-bias function g(Δτ) — FIXED (v10 sweep winner)
NUM_HEADS_VALUES = [1, 2, 4, 8, 16]   # swept, in this order; 4 = v5-v10 default (sanity check)
PIPELINE_CONFIG = {
    "Audio bin selection": "strongest-64 (N=64 frequency bins, v6 winner; TRAINING audio only, frozen for val/test)",
    "Audio time positions": "Ta = 64 per window (FIXED, v7)",
    "Sensor input":        "raw 6 IMU channels (accel x/y/z, gyro x/y/z), Ts = 128 (FIXED; no derived magnitudes)",
    "LACA beta":           f"temporal_bias_scale = {LACA_BETA} (FIXED, v9 sweep winner)",
    "LACA g(Δτ)":          f"{LAG_BIAS_FORM} — 2-layer MLP Linear(1,32)-Tanh-Linear(32,1) (FIXED, v10 sweep winner)",
    "SWEPT VAR num_heads": f"LACA attention heads ∈ {NUM_HEADS_VALUES} (d_model=256, head_dim = 256 // num_heads; "
                           f"4 = sanity check)",
    "Audio windowing":     "none (raw 1536-sample chunk -> rfft; no Hann)",
    "Audio compression":   "sqrt(magnitude)",
    "Audio normalization": "none",
    "Classification loss": "class-weighted cross-entropy (inverse TRAIN-set class frequency)",
    "Seed":                f"{SEED} (random / numpy / torch / cuda, cuDNN deterministic; reset for every head count)",
    "Everything else":     "v10 unchanged (architecture except LACA num_heads, sensor path, split, 20 epochs, AdamW, eval)",
}
print("=" * 90)
print("  AEROSYNC-MAMBA v11 — LACA NUM_HEADS SWEEP CONFIGURATION")
print("=" * 90)
for _k, _v in PIPELINE_CONFIG.items():
    print(f"  • {_k:<20}: {_v}")
print("=" * 90)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# ==============================================================================
# FIX 2: install the REAL mamba_ssm package (selective-scan SSM kernels).
#
# STRATEGY (in order):
#   1. PREBUILT WHEEL from the causal-conv1d / mamba-ssm GitHub releases, matched
#      to this exact Python / torch / CUDA / C++ ABI build. This is the ONLY
#      reliable path on Kaggle — building from source routinely hangs or OOMs
#      the kernel because Kaggle's toolchain doesn't line up with what these
#      packages' setup.py expects.
#   2. Capped from-source build (240s timeout, output captured) as a fallback
#      if no matching prebuilt wheel exists for this environment.
#   3. Pure-PyTorch fallback block (Cell 2) if both of the above fail — this is
#      the safety net that guarantees the notebook can NEVER hang/crash here,
#      but every mode in the frequency-bin sweep shares whichever backbone this
#      cell resolves to, so the 6-way comparison stays apples-to-apples either way.
# ==============================================================================
MAMBA_INSTALL_TIMEOUT_S = 240

def _detect_env_tags():
    """Python tag, torch major.minor, CUDA major.minor, C++11 ABI flag — the
    axes prebuilt causal-conv1d / mamba-ssm wheels are keyed on."""
    import sys
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    cuda_ver = torch.version.cuda  # e.g. "12.1", or None on CPU-only torch
    try:
        abi_flag = bool(torch._C._GLIBCXX_USE_CXX11_ABI)
    except Exception:
        abi_flag = bool(getattr(torch, "compiled_with_cxx11_abi", lambda: False)())
    return py_tag, torch_mm, cuda_ver, abi_flag

def _github_releases(repo, timeout_s=20):
    import urllib.request, json as _json
    url = f"https://api.github.com/repos/{repo}/releases"
    req = urllib.request.Request(url, headers={"User-Agent": "kaggle-notebook", "Accept": "application/vnd.github+json"})
    with urllib.request.urlopen(req, timeout=timeout_s) as resp:
        return _json.load(resp)

def _find_prebuilt_wheel_url(repo, py_tag, torch_mm, cuda_ver, abi_flag):
    """Best-matching .whl asset URL from `repo`'s GitHub releases, or None."""
    try:
        releases = _github_releases(repo)
    except Exception as e:
        print(f"    could not query github.com/{repo}/releases: {e!r}")
        return None

    abi_str = f"cxx11abi{'TRUE' if abi_flag else 'FALSE'}"
    cuda_major = cuda_ver.split(".")[0] if cuda_ver else None
    cuda_tag_exact = f"cu{cuda_ver.replace('.', '')}" if cuda_ver else None   # e.g. "cu121"
    exact, major_only = [], []

    for rel in releases:
        for asset in rel.get("assets", []):
            name = asset.get("name", "")
            if not name.endswith(".whl"):
                continue
            if py_tag not in name or abi_str not in name or f"torch{torch_mm}" not in name:
                continue
            if cuda_tag_exact and cuda_tag_exact in name:
                exact.append(asset["browser_download_url"])
            elif cuda_major and f"cu{cuda_major}" in name:
                major_only.append(asset["browser_download_url"])

    chosen = (exact or major_only)
    return chosen[0] if chosen else None

def _pip_run(args, timeout_s=MAMBA_INSTALL_TIMEOUT_S):
    """pip invocation that can NEVER hang or crash the kernel: hard timeout,
    output captured (not streamed — a runaway nvcc/ninja build can print
    megabytes), every exception caught."""
    import subprocess, sys
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + args
    try:
        proc = subprocess.run(cmd, timeout=timeout_s, capture_output=True, text=True, check=False)
        if proc.returncode != 0:
            tail = (proc.stderr or proc.stdout or "")[-500:]
            print(f"    ⚠️  pip install {args} failed (exit {proc.returncode}); tail:\n{tail}")
            return False
        return True
    except subprocess.TimeoutExpired:
        print(f"    ⚠️  pip install {args} exceeded {timeout_s}s — killed to avoid hanging/OOM-ing "
              f"the kernel; treating as failed.")
        return False
    except Exception as e:
        print(f"    ⚠️  pip install {args} raised {e!r}; treating as failed.")
        return False

def _install_via_prebuilt_wheel(pip_name, repo, py_tag, torch_mm, cuda_ver, abi_flag):
    url = _find_prebuilt_wheel_url(repo, py_tag, torch_mm, cuda_ver, abi_flag)
    if url is None:
        print(f"    no matching prebuilt wheel found for {pip_name} "
              f"(py={py_tag}, torch={torch_mm}, cuda={cuda_ver}, cxx11abi={abi_flag})")
        return False
    print(f"    found prebuilt wheel: {url}")
    return _pip_run([url, "--no-deps"])

_MAMBA_INSTALL_METHOD = "not attempted (no CUDA GPU)"

if torch.cuda.is_available():
    try:
        import mamba_ssm  # noqa: F401
        print("mamba_ssm already installed.")
        _MAMBA_INSTALL_METHOD = "already installed"
    except ImportError:
        py_tag, torch_mm, cuda_ver, abi_flag = _detect_env_tags()
        print(f"Environment: python={py_tag} | torch={torch_mm} | cuda={cuda_ver} | cxx11abi={abi_flag}")
        print("Installing real mamba_ssm + causal-conv1d (selective-scan SSM kernels)...")

        print("  [1/3] trying prebuilt wheels (no compilation — the reliable path on Kaggle)...")
        ok1 = _install_via_prebuilt_wheel("causal-conv1d", "Dao-AILab/causal-conv1d", py_tag, torch_mm, cuda_ver, abi_flag)
        ok2 = _install_via_prebuilt_wheel("mamba-ssm", "state-spaces/mamba", py_tag, torch_mm, cuda_ver, abi_flag) if ok1 else False

        if ok1 and ok2:
            _MAMBA_INSTALL_METHOD = "prebuilt wheel"
        else:
            print(f"  [2/3] no complete prebuilt-wheel match — falling back to a capped "
                  f"from-source build ({MAMBA_INSTALL_TIMEOUT_S}s timeout per package)...")
            # --no-build-isolation avoids re-resolving/reinstalling torch inside an isolated
            # build env, a common cause of causal-conv1d's build spinning forever on Kaggle.
            ok1b = _pip_run(["causal-conv1d>=1.2.0", "--no-build-isolation"])
            ok2b = _pip_run(["mamba-ssm", "--no-build-isolation"]) if ok1b else False
            _MAMBA_INSTALL_METHOD = "source build" if (ok1b and ok2b) else "failed"

        print(f"  [3/3] resolution: {_MAMBA_INSTALL_METHOD}")
        if _MAMBA_INSTALL_METHOD in ("prebuilt wheel", "source build"):
            print(f"✅ causal-conv1d + mamba-ssm installed via {_MAMBA_INSTALL_METHOD}.")
        else:
            print("⚠️  mamba_ssm install did not complete — the pure-PyTorch FALLBACK block will be "
                  "used (see Cell 2 / Cell 6 output). This does not affect the frequency-bin sweep: "
                  "every mode uses the same backbone, so the 6-way comparison is still apples-to-apples.")
else:
    print("⚠️  No CUDA GPU detected — mamba_ssm requires CUDA, so the pure-PyTorch FALLBACK block will be used (see Cell 2 / Cell 6 output).")


# ==============================================================================
# GLOBAL SEEDING — python `random`, numpy, torch CPU + every CUDA device, plus
# deterministic cuDNN. The mission split in Cell 4 keeps its own
# random.seed(42); this covers everything else (weight init, dropout, DataLoader
# shuffle order). set_all_seeds() is called again right before model
# construction for every condition of the LACA num_heads sweep (Cell 6).
# Note: mamba_ssm's CUDA selective-scan backward uses atomic adds, so GPU runs
# can still differ in the last few decimal places even with identical seeds.
# ==============================================================================
def set_all_seeds(seed: int = SEED) -> int:
    """Reset python / numpy / torch (CPU + all CUDA devices) RNGs to `seed`."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    return seed

set_all_seeds(SEED)
print(f"✅ Seeds set -> random={SEED}, numpy={SEED}, torch={SEED}, cuda={SEED} (cuDNN deterministic)")

  AEROSYNC-MAMBA v11 — LACA NUM_HEADS SWEEP CONFIGURATION
  • Audio bin selection : strongest-64 (N=64 frequency bins, v6 winner; TRAINING audio only, frozen for val/test)
  • Audio time positions: Ta = 64 per window (FIXED, v7)
  • Sensor input        : raw 6 IMU channels (accel x/y/z, gyro x/y/z), Ts = 128 (FIXED; no derived magnitudes)
  • LACA beta           : temporal_bias_scale = 1.0 (FIXED, v9 sweep winner)
  • LACA g(Δτ)          : mlp — 2-layer MLP Linear(1,32)-Tanh-Linear(32,1) (FIXED, v10 sweep winner)
  • SWEPT VAR num_heads : LACA attention heads ∈ [1, 2, 4, 8, 16] (d_model=256, head_dim = 256 // num_heads; 4 = sanity check)
  • Audio windowing     : none (raw 1536-sample chunk -> rfft; no Hann)
  • Audio compression   : sqrt(magnitude)
  • Audio normalization : none
  • Classification loss : class-weighted cross-entropy (inverse TRAIN-set class frequency)
  • Seed                : 42 (random / numpy / torch / cuda, cuDNN deterministic; reset for every head count)
  • Ever

# Cell 2: Bi-Modal AeroSync-Mamba Architecture (HTT + LACA + Mamba)

In [2]:
# --- Fusion backbone resolution: REAL mamba_ssm (CUDA selective-scan) vs. pure-PyTorch FALLBACK ---
_MAMBA_CLASS = None
_FUSION_BACKBONE_REASON = None
try:
    if torch.cuda.is_available():
        from mamba_ssm import Mamba as _MAMBA_CLASS
        _FUSION_BACKBONE_REASON = "mamba_ssm.Mamba (REAL selective-scan SSM: hidden state h_t, learned A/B/C, discretized Δ) — CUDA GPU detected"
    else:
        _FUSION_BACKBONE_REASON = "FALLBACK (VectorizedSSMBlock, pure PyTorch, NO SSM recurrence) — no CUDA GPU detected; mamba_ssm requires CUDA"
except ImportError as _mamba_import_err:
    _FUSION_BACKBONE_REASON = f"FALLBACK (VectorizedSSMBlock, pure PyTorch, NO SSM recurrence) — mamba_ssm import failed: {_mamba_import_err}"

print(f"\n{'='*70}\nFUSION BACKBONE RESOLUTION: {_FUSION_BACKBONE_REASON}\n{'='*70}\n")


# --- 1. Continuous Temporal Positional Embedding & HTT ---
class ContinuousTimeEmbedding(nn.Module):
    def __init__(self, d_model: int, num_frequencies: int = 64):
        super().__init__()
        self.frequencies = nn.Parameter(
            torch.exp(torch.linspace(0, math.log(1000.0), num_frequencies)), requires_grad=False
        )
        self.mlp = nn.Sequential(
            nn.Linear(2 * num_frequencies, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tau: torch.Tensor) -> torch.Tensor:
        if tau.dim() == 2:
            tau = tau.unsqueeze(-1)
        angles = tau * self.frequencies * 2.0 * math.pi
        fourier = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)
        return self.mlp(fourier)


class HeterogeneousTemporalTokenization(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.d_model = d_model
        # 0: Audio, 1: Sensor Telemetry
        self.modality_embeddings = nn.Embedding(2, d_model)
        self.time_embedding = ContinuousTimeEmbedding(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z_bar: torch.Tensor, tau: torch.Tensor, modality_idx: int) -> torch.Tensor:
        mod_idx = torch.tensor(modality_idx, device=z_bar.device, dtype=torch.long)
        e_m = self.modality_embeddings(mod_idx).view(1, 1, self.d_model)
        phi_tau = self.time_embedding(tau)
        return self.norm(z_bar + e_m + phi_tau)


# --- 2. Bi-Modal LACA (Audio <-> Telemetry Alignment) ---
# v10: the lag-bias function g(Δτ) is selectable. Every form maps Δτ of shape
# (B, T_m, T_n, 1) to a bias of the SAME shape (B, T_m, T_n, 1), exactly like the
# v5-v9 MLP, so PairwiseLACA.forward is unchanged:
#   softmax(scores + beta * g(Δτ).squeeze(-1).unsqueeze(1))
LAG_BIAS_FORMS = ("mlp", "rbf", "bucketed")
RBF_N_CENTERS = 16
N_LAG_BUCKETS = 32


class RBFLagBias(nn.Module):
    """g(Δτ) = Linear_16->1( [exp(-(Δτ - c_k)^2 / (2σ^2))]_k ).
    16 centers evenly spaced over the TRAINING-set Δτ range, σ = spacing between
    adjacent centers. Centers and σ are fixed buffers (not learned); only the
    16->1 linear combination (weights + bias) is learned."""
    def __init__(self, lag_range, n_centers: int = RBF_N_CENTERS):
        super().__init__()
        lo, hi = float(lag_range[0]), float(lag_range[1])
        assert hi > lo, f"RBF lag range must be non-empty, got {lag_range}"
        centers = torch.linspace(lo, hi, n_centers)
        self.register_buffer("centers", centers)
        self.register_buffer("sigma", torch.tensor((hi - lo) / (n_centers - 1)))
        self.linear = nn.Linear(n_centers, 1)

    def forward(self, delta_tau: torch.Tensor) -> torch.Tensor:
        # (..., 1) - (16,) -> (..., 16) basis activations -> (..., 1)
        phi = torch.exp(-(delta_tau - self.centers) ** 2 / (2.0 * self.sigma ** 2))
        return self.linear(phi)


class BucketedLagBias(nn.Module):
    """g(Δτ) = table[bucket(Δτ)] — T5-style bucketed relative bias.
    32 fixed-width buckets spanning the TRAINING-set Δτ range (33 edges), one
    learnable scalar per bucket, no interpolation. Δτ outside the training range
    is clamped to the first / last bucket. The table starts at zero (no bias)."""
    def __init__(self, lag_range, n_buckets: int = N_LAG_BUCKETS):
        super().__init__()
        lo, hi = float(lag_range[0]), float(lag_range[1])
        assert hi > lo, f"bucket lag range must be non-empty, got {lag_range}"
        self.n_buckets = n_buckets
        self.register_buffer("edges", torch.linspace(lo, hi, n_buckets + 1))
        self.bias_table = nn.Embedding(n_buckets, 1)
        nn.init.zeros_(self.bias_table.weight)

    def bucket_index(self, delta_tau: torch.Tensor) -> torch.Tensor:
        # Only the 31 INNER edges are searched: bucket k = [edges[k], edges[k+1]),
        # anything below edges[1] -> 0 and anything at/above edges[-2] -> 31, which is
        # exactly the clamp-to-first/last-bucket rule for out-of-range Δτ.
        return torch.bucketize(delta_tau.squeeze(-1).contiguous(), self.edges[1:-1].contiguous(), right=True)

    def forward(self, delta_tau: torch.Tensor) -> torch.Tensor:
        return self.bias_table(self.bucket_index(delta_tau))   # (..., 1)


def build_lag_bias(lag_bias_form: str, lag_range=None) -> nn.Module:
    assert lag_bias_form in LAG_BIAS_FORMS, f"unknown lag_bias_form {lag_bias_form!r}; expected one of {LAG_BIAS_FORMS}"
    # The v5-v9 MLP is ALWAYS constructed first, so every form consumes exactly the
    # same torch RNG draws here. Result: every OTHER module of the model gets
    # bit-identical initial weights in all 3 conditions, and the RNG stream after
    # construction (DataLoader shuffle order, dropout masks) is identical too —
    # only g(Δτ) differs. For "mlp" this is literally the v9 line, so its RNG use
    # and parameter order are unchanged from v9.
    mlp = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 1))
    if lag_bias_form == "mlp":
        return mlp
    assert lag_range is not None, f"lag_bias_form={lag_bias_form!r} needs the training-set Δτ range"
    with torch.random.fork_rng(devices=[]):   # init draws here do not advance the global CPU RNG
        return RBFLagBias(lag_range) if lag_bias_form == "rbf" else BucketedLagBias(lag_range)


class PairwiseLACA(nn.Module):
    def __init__(self, d_model: int, num_heads: int = 4, beta: float = 1.0,
                 lag_bias_form: str = "mlp", lag_range=None):
        super().__init__()
        self.d_model = d_model
        assert d_model % num_heads == 0, (
            f"d_model={d_model} must be divisible by num_heads={num_heads} for an integer head_dim")
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.beta = beta

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.lag_bias_form = lag_bias_form
        self.lag_bias = build_lag_bias(lag_bias_form, lag_range)

    def forward(self, H_m, H_n, tau_m, tau_n):
        B, T_m, _ = H_m.shape
        _, T_n, _ = H_n.shape
        Q = self.w_q(H_m).view(B, T_m, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        delta_tau = (tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)
        B_mn = self.lag_bias(delta_tau).squeeze(-1).unsqueeze(1)

        attn_weights = F.softmax(scores + self.beta * B_mn, dim=-1)
        out = torch.matmul(attn_weights, V).transpose(1, 2).contiguous().view(B, T_m, self.d_model)
        H_tilde_mn = self.out_proj(out)

        A_mn = attn_weights.mean(dim=1)
        tau_diff = tau_n.unsqueeze(1) - tau_m.unsqueeze(2)
        expected_lag = (A_mn * tau_diff).sum(dim=(1, 2)) / float(T_m)
        return H_tilde_mn, A_mn, expected_lag


class BiModalLACAFusion(nn.Module):
    def __init__(self, d_model: int, num_heads: int = 4, beta: float = 1.0,
                 lag_bias_form: str = "mlp", lag_range=None):
        super().__init__()
        self.laca_as = PairwiseLACA(d_model, num_heads, beta, lag_bias_form, lag_range)
        self.laca_sa = PairwiseLACA(d_model, num_heads, beta, lag_bias_form, lag_range)
        self.gamma_as = nn.Parameter(torch.tensor(0.5))
        self.gamma_sa = nn.Parameter(torch.tensor(0.5))

    def forward(self, Ha, Hs, tau_a, tau_s):
        H_a_from_s, _, lag_as = self.laca_as(Ha, Hs, tau_a, tau_s)
        H_s_from_a, _, lag_sa = self.laca_sa(Hs, Ha, tau_s, tau_a)

        H_tilde_a = Ha + self.gamma_as * H_a_from_s
        H_tilde_s = Hs + self.gamma_sa * H_s_from_a
        return H_tilde_a, H_tilde_s, {"lag_as": lag_as, "lag_sa": lag_sa}


# --- 3a. FALLBACK: pure-PyTorch gated-conv block (NOT a real SSM) ---
# Used ONLY when mamba_ssm / CUDA is unavailable (see resolution printed
# above). This block has NO hidden state h_t, NO A/B/C matrices, and NO
# discretization step — it is a depthwise-conv + gated-MLP block, kept only
# as a last-resort fallback so the notebook still runs on CPU-only machines.
class VectorizedSSMBlock(nn.Module):
    def __init__(self, d_model: int, d_conv: int = 4, expand: int = 2):
        super().__init__()
        self.d_inner = int(expand * d_model)
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner, kernel_size=d_conv,
            padding=d_conv - 1, groups=self.d_inner, bias=True
        )
        self.mixer = nn.Sequential(
            nn.Linear(self.d_inner, self.d_inner),
            nn.GELU(),
            nn.Linear(self.d_inner, self.d_inner)
        )
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, D = x.shape
        residual = x
        x_norm = self.norm(x)
        xz = self.in_proj(x_norm)
        x_branch, z_branch = xz.chunk(2, dim=-1)

        x_conv = self.conv1d(x_branch.transpose(1, 2))[:, :, :L].transpose(1, 2)
        x_conv = F.silu(x_conv)

        y = self.mixer(x_conv) * F.silu(z_branch)
        return residual + self.out_proj(y)


# --- 3b. REAL Mamba block (genuine selective-scan SSM) ---
# Thin wrapper around mamba_ssm.Mamba, which implements the actual S6
# recurrence (hidden state h_t, learned continuous A/B/C matrices, and a
# discretization step Δ), wrapped in the same pre-norm + residual pattern as
# the fallback block above so it is a drop-in replacement.
class RealMambaBlock(nn.Module):
    def __init__(self, d_model: int, d_conv: int = 4, expand: int = 2, d_state: int = 16):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = _MAMBA_CLASS(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.mamba(self.norm(x))


# --- 4. Full Bi-Modal AeroSync-Mamba Model ---
class BiModalAeroSyncMamba(nn.Module):
    def __init__(
        self,
        d_model: int = 256,
        n_classes: int = 5,
        num_mamba_blocks: int = 4,
        dropout: float = 0.15,
        temporal_bias_scale: float = 1.0,
        audio_in_dim: int = 64,
        sensor_in_dim: int = 6,
        lag_bias_form: str = "mlp",
        lag_range=None,
        laca_num_heads: int = 4,
    ):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes

        # Sensor 1D-CNN encoder
        self.sensor_encoder = nn.Sequential(
            nn.Conv1d(sensor_in_dim, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=3, padding=1), nn.BatchNorm1d(d_model), nn.GELU(),
        )
        # Audio feature projector
        self.audio_encoder = nn.Sequential(
            nn.Linear(audio_in_dim, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Linear(d_model, d_model)
        )

        self.htt = HeterogeneousTemporalTokenization(d_model)
        self.lag_bias_form = lag_bias_form
        self.laca = BiModalLACAFusion(d_model=d_model, num_heads=laca_num_heads, beta=temporal_bias_scale,
                                      lag_bias_form=lag_bias_form, lag_range=lag_range)
        self.sep_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        # --- FIX 2: use the REAL mamba_ssm block when available, otherwise the
        # pure-PyTorch fallback (see resolution printed at the top of this cell,
        # and the unmissable print in Cell 6 right after model construction). ---
        if _MAMBA_CLASS is not None:
            self.mamba_blocks = nn.ModuleList([
                RealMambaBlock(d_model=d_model, d_conv=4, expand=2) for _ in range(num_mamba_blocks)
            ])
        else:
            self.mamba_blocks = nn.ModuleList([
                VectorizedSSMBlock(d_model=d_model, d_conv=4, expand=2) for _ in range(num_mamba_blocks)
            ])
        self.fusion_backbone_reason = _FUSION_BACKBONE_REASON

        self.pool_attn = nn.Sequential(nn.Linear(d_model, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, n_classes),
        )

        self.aux_a = nn.Linear(d_model, n_classes)
        self.aux_s = nn.Linear(d_model, n_classes)

    def forward(self, xa, xs, ta, ts):
        B = xa.shape[0]

        # 1. Modality Encoders
        za = self.audio_encoder(xa)
        zs = self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)

        # 2. HTT (0: Audio, 1: Sensor)
        Ha = self.htt(za, ta, modality_idx=0)
        Hs = self.htt(zs, ts, modality_idx=1)

        # 3. Bi-Modal LACA Alignment
        H_tilde_a, H_tilde_s, aux = self.laca(Ha, Hs, ta, ts)
        za_p = H_tilde_a.mean(dim=1)
        zs_p = H_tilde_s.mean(dim=1)

        # 4. Cross-Modal Mamba Fusion: [ H_tilde_a ; [SEP] ; H_tilde_s ]
        sep = self.sep_token.expand(B, -1, -1)
        H_fused = torch.cat([H_tilde_a, sep, H_tilde_s], dim=1)

        F_seq = H_fused
        for blk in self.mamba_blocks:
            F_seq = blk(F_seq)

        weights = F.softmax(self.pool_attn(F_seq), dim=1)
        zf = (F_seq * weights).sum(dim=1)
        logits = self.classifier(zf)

        pooled_reps = {"z_a": za_p, "z_s": zs_p}
        aux_preds = {"y_a": self.aux_a(za_p), "y_s": self.aux_s(zs_p)}
        return logits, pooled_reps, aux_preds, aux

print("Bi-Modal AeroSync-Mamba model class defined successfully!")


FUSION BACKBONE RESOLUTION: mamba_ssm.Mamba (REAL selective-scan SSM: hidden state h_t, learned A/B/C, discretized Δ) — CUDA GPU detected

Bi-Modal AeroSync-Mamba model class defined successfully!


# Cell 3: Bi-Modal Loss Function (Classification + Audio-IMU InfoNCE + KL)

In [3]:
class BiModalAeroSyncLoss(nn.Module):
    def __init__(self, lambda_align=0.1, lambda_cons=0.1, temperature=0.07, class_weights=None):
        super().__init__()
        self.lambda_align = lambda_align
        self.lambda_cons = lambda_cons
        self.temperature = temperature
        # --- FIX 4: class-weighted classification loss. `class_weights` is an
        # inverse-frequency tensor computed from REAL training-set class counts
        # in Cell 6 (printed there for verification), not hardcoded here. ---
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def _infonce(self, z1, z2):
        z1_norm = F.normalize(z1, p=2, dim=-1)
        z2_norm = F.normalize(z2, p=2, dim=-1)
        sim = torch.matmul(z1_norm, z2_norm.t()) / self.temperature
        labels = torch.arange(z1.shape[0], device=z1.device)
        return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.t(), labels))

    def forward(self, logits, targets, pooled, aux_preds):
        # 1. Fault Classification Loss
        loss_cls = self.ce(logits, targets)

        # 2. Audio-Sensor Contrastive Alignment Loss
        loss_align = self._infonce(pooled["z_a"], pooled["z_s"])

        # 3. Auxiliary Consistency Loss
        p_fused = F.softmax(logits.detach(), dim=-1)
        loss_cons = (
            F.kl_div(F.log_softmax(aux_preds["y_a"], dim=-1), p_fused, reduction="batchmean") +
            F.kl_div(F.log_softmax(aux_preds["y_s"], dim=-1), p_fused, reduction="batchmean")
        ) / 2.0

        total = loss_cls + self.lambda_align * loss_align + self.lambda_cons * loss_cons
        return total, {"loss_cls": float(loss_cls.item()), "loss_align": float(loss_align.item())}

def compute_all_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    try:
        auroc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except Exception:
        auroc = float("nan")
    return {"accuracy": acc, "macro_f1": f1, "balanced_accuracy": bal_acc, "macro_auroc": auroc}

print("Loss & Metrics helper ready!")

Loss & Metrics helper ready!


# Cell 4 (Self-Downloading & Preprocessing): Real TII UAV Dataset


In [4]:
import os
import json
import random
import numpy as np
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
import torch

DATASET_BASE = Path("/kaggle/working/tii_uav_dataset")
DATASET_ROOT = DATASET_BASE / "Dataset"
CACHE_DIR = Path("/kaggle/working/preprocessed_bimodal_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# 1. Automatically clone the official repository if not present
if not DATASET_ROOT.exists():
    print("📥 Cloning official TII UAV Realistic Fault Dataset from GitHub...")
    !git clone --depth 1 https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git /kaggle/working/tii_uav_dataset
    print("✅ Dataset cloned successfully!")

_chunk_duration_log = []
_fft_shape_printed = False  # verification: print the FFT feature shape once

# Three different sizes — kept as separate named constants so they are never confused:
N_AUDIO_BINS = 64             # audio FREQUENCY axis: strongest-64 bins = feature dim per time step (FIXED, v6 winner)
TA_AUDIO = 64                 # audio TIME axis: time positions per window (FIXED, v7)
TS_SENSOR = 128               # sensor TIME axis: time positions per window (FIXED, unchanged from v6)
_audio_chunks_per_window = []  # verification: real audio chunks available in each window (vs Ta)
_MISSION_CACHE = {}  # mission_dir -> parse_real_mission(...) output, parsed ONCE and reused for every N

# 2. Real Flight Mission Parser (6-Axis IMU + Audio)
def parse_real_mission(mission_dir: Path):
    sensor_matches = list(mission_dir.glob("*SensorCombined*.jsonl"))
    audio_matches  = list(mission_dir.glob("*AudioBuffer*.jsonl"))
    if not sensor_matches or not audio_matches:
        return None

    # 2.1 Parse Real IMU
    sensor_times, accel_data, gyro_data = [], [], []
    with open(sensor_matches[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            try:
                rec = json.loads(line)
                t_sec = float(rec["timestamp"]) / 1e6
                sensor_times.append(t_sec)
                accel_data.append(rec.get("accelerometer_m_s2", [0.0, 0.0, 9.81]))
                gyro_data.append(rec.get("gyro_rad", [0.0, 0.0, 0.0]))
            except Exception:
                continue

    if len(sensor_times) < 50: return None
    sensor_times = np.array(sensor_times, dtype=np.float64)
    sensor_times = sensor_times - sensor_times[0]
    telemetry = np.concatenate([np.array(accel_data, dtype=np.float32), np.array(gyro_data, dtype=np.float32)], axis=-1)

    # 2.2 Parse Real Audio
    audio_chunks = []
    with open(audio_matches[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            try:
                rec = json.loads(line)
                samples = rec.get("audio_data", rec.get("buffer", rec.get("data", [])))
                if len(samples) > 0: audio_chunks.append(samples)
            except Exception:
                continue

    if len(audio_chunks) < 10: return None
    audio_chunks = np.array(audio_chunks, dtype=np.float32)

    # --- FIX: derive REAL per-chunk audio timestamps ---
    # `recording_started_time` is a constant repeated on every line (not a
    # real per-chunk timestamp), and the previous fixed 0.10s/chunk guess
    # doesn't match the data (empirically ~17-19ms/chunk). Instead, derive
    # chunk duration from this mission's REAL, IMU-verified flight duration,
    # since audio and IMU are recorded within the same continuous rosbag
    # session.
    mission_duration_s = sensor_times[-1] - sensor_times[0]  # already zero-based
    n_chunks = len(audio_chunks)
    chunk_duration_s = mission_duration_s / n_chunks
    audio_times = np.arange(n_chunks, dtype=np.float64) * chunk_duration_s

    global _chunk_duration_log
    _chunk_duration_log.append(chunk_duration_s)

    return {
        "sensor_times": sensor_times,
        "telemetry": telemetry,
        "audio_times": audio_times,
        "audio_chunks": audio_chunks,
    }


def get_mission(mission_dir: Path):
    """parse_real_mission, memoized: every mission's JSONL is parsed once and the
    same arrays are reused for every N (identical to re-parsing each time)."""
    if mission_dir not in _MISSION_CACHE:
        _MISSION_CACHE[mission_dir] = parse_real_mission(mission_dir)
    return _MISSION_CACHE[mission_dir]


# 2.3 Strongest-N FFT bin selection (TRAINING-SET audio only, unsupervised, leakage-safe)
def select_strongest_audio_bins(train_mission_dirs, n_bins):
    """
    Selects the `n_bins` FFT magnitude bins with the highest mean magnitude,
    computed ONLY from training-set audio (raw waveform energy — never class
    labels). Called once per N, before any train/val/test window slicing for
    that N, so the exact same frozen indices are reused identically across all
    three splits.

    Identical criterion to the v5 strongest-64 selection; only the budget N is
    a parameter now. Returns (sorted bin indices, data-derived bin width in Hz
    or None).
    """
    print(f"[STRONGEST-{n_bins}] Computing bin-selection statistics from TRAINING-SET audio only "
          f"({len(train_mission_dirs)} training missions) — BEFORE any train/validation/test "
          "window preprocessing.")

    bin_magnitude_sum = None
    bin_count = 0
    fft_len_ref = None
    local_chunk_durations = []

    for m_dir in tqdm(train_mission_dirs, desc=f"Bin selection N={n_bins} (train-only audio scan)", leave=False):
        mission_data = get_mission(m_dir)
        if mission_data is None:
            continue
        audio = mission_data["audio_chunks"]
        a_times = mission_data["audio_times"]
        if len(a_times) > 1:
            local_chunk_durations.append((a_times[-1] - a_times[0]) / (len(a_times) - 1))

        mag = np.abs(np.fft.rfft(audio, axis=1))  # (n_chunks, fft_len), unsupervised energy
        if fft_len_ref is None:
            fft_len_ref = mag.shape[1]
        min_len = min(fft_len_ref, mag.shape[1])
        mag = mag[:, :min_len]
        if bin_magnitude_sum is not None and len(bin_magnitude_sum) > min_len:
            bin_magnitude_sum = bin_magnitude_sum[:min_len]
        fft_len_ref = min_len

        chunk_sum = mag.sum(axis=0)
        bin_magnitude_sum = chunk_sum if bin_magnitude_sum is None else bin_magnitude_sum + chunk_sum
        bin_count += mag.shape[0]

    assert bin_magnitude_sum is not None and bin_count > 0, (
        f"STRONGEST-{n_bins} FAILED: no training-set audio chunks were available for bin selection."
    )

    mean_magnitude = bin_magnitude_sum / bin_count  # unsupervised magnitude/energy criterion
    top_bins = np.sort(np.argsort(mean_magnitude)[::-1][:n_bins]).astype(int)

    assert len(top_bins) == n_bins, (
        f"STRONGEST-{n_bins} FAILED: selected {len(top_bins)} bins, expected {n_bins}"
    )

    # --- Hz labeling ---
    # This notebook does not document a raw microphone sample rate anywhere,
    # so we estimate an effective sample rate from quantities the notebook
    # already derives from real data: samples-per-audio-buffer (inverted from
    # the rFFT length) and the real per-mission inter-buffer timing computed
    # above (same derivation `parse_real_mission` already uses for
    # `chunk_duration_s`). This is a data-derived estimate used only for
    # human-readable labeling below — it never affects which bins are
    # selected or how features are built.
    samples_per_buffer = fft_len_ref * 2 - 2
    est_sample_rate_hz = (
        samples_per_buffer / np.mean(local_chunk_durations) if local_chunk_durations else None
    )
    bin_width_hz = est_sample_rate_hz / samples_per_buffer if est_sample_rate_hz else None

    print(f"[STRONGEST-{n_bins}] Selected {n_bins} bins from {bin_count} training-set audio chunks "
          f"across {len(train_mission_dirs)} missions (FFT length = {fft_len_ref}).")
    print(f"[STRONGEST-{n_bins}] Selected bin indices (ascending): {top_bins.tolist()}")
    if bin_width_hz:
        print(f"[STRONGEST-{n_bins}] Estimated audio sample rate ≈ {est_sample_rate_hz:.1f} Hz "
              "(derived from audio buffer length and real inter-buffer timing across training "
              "missions; the dataset does not document a mic sample rate directly, so this is "
              "a data-derived estimate for labeling only, not a hardcoded spec).")
        # contiguous runs of selected bins -> approx. frequency bands
        runs, start = [], top_bins[0]
        for prev, cur in zip(top_bins[:-1], top_bins[1:]):
            if cur != prev + 1:
                runs.append((start, prev))
                start = cur
        runs.append((start, top_bins[-1]))
        print(f"[STRONGEST-{n_bins}] Selected bins as {len(runs)} contiguous band(s) -> approx. frequency (Hz):")
        for lo, hi in runs:
            print(f"    bins {lo:3d}-{hi:3d} ({hi - lo + 1:3d} bins)  ->  {lo * bin_width_hz:8.1f} - {hi * bin_width_hz:8.1f} Hz")
    else:
        print(f"[STRONGEST-{n_bins}] Could not estimate a sample rate (insufficient timing data); "
              "printing bin indices only.")

    return top_bins, bin_width_hz


# 2.4 Audio feature extraction — the single, permanent path (v5), width = N:
#     raw chunk -> rfft (no window) -> |.| -> frozen strongest-N bins -> sqrt
#     (no normalization).
def compute_audio_features(audio_chunks, bin_indices):
    """
    audio_chunks : (n_chunks, chunk_len) raw waveform chunks of one mission
    bin_indices  : frozen strongest-N bin indices from select_strongest_audio_bins
                   (computed once per N from TRAINING audio; never recomputed here)
    returns      : (n_chunks, N) float32 = sqrt(|rfft|) at those bins, N = len(bin_indices)
    """
    assert bin_indices is not None and len(bin_indices) > 0, (
        "FEATURES FAILED: strongest-N bin indices must be computed from training audio "
        "(select_strongest_audio_bins) before any feature extraction."
    )
    n_bins = len(bin_indices)

    audio_fft_mag = np.abs(np.fft.rfft(audio_chunks, axis=1))  # (n_chunks, chunk_len//2 + 1), REAL FFT
    n_chunks_mission = audio_fft_mag.shape[0]
    n_bins_available = audio_fft_mag.shape[1]
    valid_idx = bin_indices[bin_indices < n_bins_available]

    a_feats = np.zeros((n_chunks_mission, n_bins), dtype=np.float32)
    a_feats[:, :len(valid_idx)] = np.sqrt(audio_fft_mag[:, valid_idx]).astype(np.float32)

    assert a_feats.shape == (n_chunks_mission, n_bins), (
        f"FEATURES FAILED: audio feature array shape is {a_feats.shape}, "
        f"expected (n_chunks={n_chunks_mission}, N={n_bins})"
    )
    return a_feats


# 3. Slicing into 2.56s Diagnostic Samples
def slice_real_bimodal(mission_data, class_label, bin_indices, Ta, window_sec=2.56, step_sec=1.28, Ts=128):
    # Ta = audio TIME positions per window (required, no default); Ts = sensor TIME positions (unchanged).
    # The frequency width of every audio time step is n_bins = len(bin_indices), independent of Ta.
    global _fft_shape_printed
    s_times = mission_data["sensor_times"]
    telemetry = mission_data["telemetry"]
    a_times = mission_data["audio_times"]
    audio = mission_data["audio_chunks"]

    assert bin_indices is not None and len(bin_indices) > 0, (
        "SLICE FAILED: strongest-N bin indices must be computed from training audio "
        "(select_strongest_audio_bins) before any window preprocessing."
    )

    # --- strongest-N audio features: FFT magnitude spectrum, frozen top-N bins ---
    # Previously: the first 64 FFT bins (low frequencies only) were used
    # unconditionally for every split. Now: the 64 bins with the highest mean
    # magnitude in TRAINING-set audio (computed once in
    # select_strongest_audio_bins, unsupervised, leakage-safe) are frozen and
    # reused identically for train, validation, and test.
    # FFT (no window) -> frozen strongest-N bins -> sqrt, via compute_audio_features (2.4).
    n_bins = len(bin_indices)
    a_feats = compute_audio_features(audio, bin_indices)
    n_chunks_mission = audio.shape[0]

    assert a_feats.shape == (n_chunks_mission, n_bins), (
        f"STRONGEST-{n_bins} FAILED: audio feature array shape is {a_feats.shape}, "
        f"expected (n_chunks={n_chunks_mission}, {n_bins})"
    )

    if not _fft_shape_printed:
        print(f"[STRONGEST-{n_bins} VERIFICATION] audio chunk shape (raw waveform): {audio.shape} "
              f"| np.fft.rfft output shape: {(audio.shape[0], audio.shape[1] // 2 + 1)} "
              f"| final a_feats (sqrt-compressed, frozen strongest-{n_bins} bins) shape: {a_feats.shape}")
        print(f"[STRONGEST-{n_bins} VERIFICATION] Confirmed: features were produced via np.fft.rfft "
              f"(frequency-domain magnitude spectrum) using the frozen strongest-{n_bins} bin indices, "
              f"NOT the first {n_bins} bins.")
        _fft_shape_printed = True

    max_time = min(s_times[-1], a_times[-1])
    samples = []
    t_start = 0.0

    while t_start + window_sec <= max_time:
        t_end = t_start + window_sec
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < 10:
            t_start += step_sec
            continue

        # --- FIX 1: REAL per-window sensor timestamps ---
        # Previously: target_s_t = np.linspace(t_start, t_end, Ts) — a synthetic
        # uniform grid that is IDENTICAL for every window (window_sec is fixed),
        # so nothing real ever reached the model downstream. Now: pick Ts
        # evenly-spaced INDICES into the actual raw IMU sample timestamps that
        # fall inside this window. Real IMU logging is not perfectly uniform, so
        # these timestamps genuinely vary window-to-window and mission-to-mission.
        s_idx_in_window = np.where(s_mask)[0]
        real_s_pos = np.linspace(0, len(s_idx_in_window) - 1, Ts).round().astype(int)
        real_s_idx = s_idx_in_window[real_s_pos]
        target_s_t = s_times[real_s_idx]  # REAL raw sensor timestamps (seconds, mission-relative)

        s_window = np.zeros((Ts, 6), dtype=np.float32)
        for c in range(6):
            s_window[:, c] = np.interp(target_s_t, s_times[s_mask], telemetry[s_mask, c])

        # --- FIX 1: REAL per-window audio timestamps ---
        # audio_times itself is a per-mission uniform grid derived from the
        # mission's real chunk_duration_s (see parse_real_mission above), which
        # genuinely varies mission-to-mission (see the chunk-duration min/max
        # printed at the end of this cell). We pick real indices from it within
        # this window rather than an independent synthetic linspace.
        a_mask = (a_times >= t_start) & (a_times <= t_end)
        _audio_chunks_per_window.append(int(a_mask.sum()))
        if a_mask.sum() >= 2:
            a_idx_in_window = np.where(a_mask)[0]
            real_a_pos = np.linspace(0, len(a_idx_in_window) - 1, Ta).round().astype(int)
            real_a_idx = a_idx_in_window[real_a_pos]
            target_a_t = a_times[real_a_idx]  # REAL per-mission audio timestamps
        else:
            # Fallback only for the rare window with too few real audio samples
            target_a_t = np.linspace(t_start, t_end, Ta)

        a_window = np.zeros((Ta, n_bins), dtype=np.float32)  # (TIME positions, FREQUENCY bins)
        for c in range(n_bins):
            a_window[:, c] = np.interp(target_a_t, a_times, a_feats[:, c], left=0.0, right=0.0)

        assert a_window.shape == (Ta, n_bins), (
            f"STRONGEST-{n_bins} FAILED: window audio feature shape is {a_window.shape}, "
            f"expected ({Ta}, {n_bins})"
        )

        samples.append({
            "x_a": a_window,
            "x_s": s_window,
            "label": int(class_label),
            "tau_a": (target_a_t - t_start).astype(np.float32),  # REAL, zero-based, seconds
            "tau_s": (target_s_t - t_start).astype(np.float32),  # REAL, zero-based, seconds
            "window_sec": np.float32(window_sec),
        })
        t_start += step_sec

    return samples

# --- VERIFICATION: no leftover mode-switching in the audio feature path ---
import inspect
_feat_params = list(inspect.signature(compute_audio_features).parameters)
_slice_params = list(inspect.signature(slice_real_bimodal).parameters)
_switch_names = ("window_fn", "compression", "norm_stats", "normalization", "audio_mode", "mode")
_switch_params = [p for p in _feat_params + _slice_params if p in _switch_names]
try:
    _feat_src = inspect.getsource(compute_audio_features)
    _dead_branches = [t for t in ("hanning", "log1p", "log10", "zscore", "window_fn", "compression ==",
                                  "norm_stats", "elif") if t in _feat_src]
except (OSError, TypeError):
    _feat_src, _dead_branches = None, []
_stale_globals = [n for n in ("AUDIO_WINDOW_FNS", "AUDIO_COMPRESSIONS", "LOG_DB_EPS", "get_audio_window",
                              "compute_train_norm_stats", "N_BUDGETS", "TA_VALUES", "SENSOR_CHANNEL_CONFIGS",
                              "DERIVED_CHANNEL_SOURCES") if n in globals()]

assert _feat_params == ["audio_chunks", "bin_indices"], f"compute_audio_features has unexpected parameters: {_feat_params}"
assert not _switch_params, f"mode-switching parameters remain: {_switch_params}"
assert not _dead_branches, f"dead condition branches remain in compute_audio_features: {_dead_branches}"
print(f"[NO-MODE-SWITCH CHECK] compute_audio_features{tuple(_feat_params)} | slice_real_bimodal has no "
      f"window/compression/normalization parameters | source scan for hann/log1p/log_db/zscore/elif "
      f"branches: {'clean' if _feat_src is not None else 'source unavailable (signature check only)'} ✅")
if _stale_globals:
    print(f"⚠️  stale globals from an older notebook are still in this kernel: {_stale_globals} "
          f"(unused by v9 — restart the kernel for a clean run)")
else:
    print("[NO-MODE-SWITCH CHECK] no experiment globals (window / compression / normalization options) and no leftover N / Ta / sensor-feature globals defined ✅")

# 4. Mission-Level Stratification across all 99 Flights
class_missions = defaultdict(list)
for c_dir in sorted(DATASET_ROOT.iterdir()):
    if not c_dir.is_dir() or c_dir.name.startswith("."): continue
    try: c_label = int(c_dir.name)
    except ValueError: continue
    for m_dir in sorted(c_dir.iterdir()):
        if m_dir.is_dir() and not m_dir.name.startswith("."):
            class_missions[c_label].append(m_dir)

random.seed(42)

train_mission_map, val_mission_map, test_mission_map = {}, {}, {}
for c_label in sorted(class_missions.keys()):
    missions = class_missions[c_label]
    random.shuffle(missions)
    n_train = max(1, int(len(missions) * 0.70))
    n_val = max(1, int(len(missions) * 0.15))

    train_mission_map[c_label] = missions[:n_train]
    val_mission_map[c_label]   = missions[n_train:n_train + n_val]
    test_mission_map[c_label]  = missions[n_train + n_val:]

    print(f"Class {c_label} ({len(missions)} flights) -> "
          f"Train: {len(train_mission_map[c_label])}, "
          f"Val: {len(val_mission_map[c_label])}, "
          f"Test: {len(test_mission_map[c_label])}")

# --- 5. Strongest-64 bins: selected ONCE from TRAINING missions only, frozen ---
# The FREQUENCY axis does not depend on the audio TIME axis (Ta), so the same
# frozen indices are reused for every Ta condition and for train/val/test.
all_train_missions = [m for missions in train_mission_map.values() for m in missions]
_heldout_missions = {m for mp in (val_mission_map, test_mission_map) for ms in mp.values() for m in ms}
assert not (set(all_train_missions) & _heldout_missions), (
    "LEAKAGE: training missions overlap validation/test missions")

SELECTED_BIN_INDICES, AUDIO_BIN_WIDTH_HZ = select_strongest_audio_bins(all_train_missions, n_bins=N_AUDIO_BINS)
_bin_fingerprint_at_selection = (SELECTED_BIN_INDICES[:5].tolist(), SELECTED_BIN_INDICES[-5:].tolist())
print(f"[STRONGEST-{N_AUDIO_BINS}] frozen once from {len(all_train_missions)} TRAINING missions "
      f"(0 of {len(_heldout_missions)} validation/test missions used); reused unchanged for every Ta.")


# --- 6. Per-Ta preprocessing (called once per Ta by run_experiment in Cell 6) ---
def preprocess_for_ta(Ta):
    """Train/val/test windows with Ta audio time positions per window, using the
    frozen strongest-64 bins. Sensor side (TS_SENSOR positions, 6 channels) is
    built by exactly the same code as v6 and does not depend on Ta."""
    global _fft_shape_printed
    _fft_shape_printed = False
    _audio_chunks_per_window.clear()

    split_maps = [("TRAIN", train_mission_map), ("VAL", val_mission_map), ("TEST", test_mission_map)]
    split_samples = {}
    _fp_train = None
    for split_label, mission_map in split_maps:
        note = "(frozen strongest-64 bins)" if split_label == "TRAIN" else "(same frozen bin indices — NOT recomputed)"
        print(f"\n--- Preprocessing {split_label} split, Ta={Ta} {note} ---")
        samples = []
        for c_label in sorted(class_missions.keys()):
            for m in tqdm(mission_map[c_label], desc=f"{split_label.capitalize()} Class {c_label} (Ta={Ta})", leave=False):
                p = get_mission(m)
                if p:
                    samples.extend(slice_real_bimodal(p, c_label, SELECTED_BIN_INDICES, Ta=Ta, Ts=TS_SENSOR))
        split_samples[split_label] = samples

        _fp = (SELECTED_BIN_INDICES[:5].tolist(), SELECTED_BIN_INDICES[-5:].tolist())
        if split_label == "TRAIN":
            _fp_train = _fp
            print(f"[STRONGEST-{N_AUDIO_BINS}] Frozen bin indices applied @ TRAIN (first5, last5): {_fp} "
                  f"| matches selection: {_fp == _bin_fingerprint_at_selection}")
        else:
            print(f"[STRONGEST-{N_AUDIO_BINS}] Frozen bin indices applied @ {split_label:<5} (first5, last5): {_fp} "
                  f"| matches TRAIN: {_fp == _fp_train}")

    # --- Shape verification across all three splits:
    #     audio  (n_windows, Ta, N_AUDIO_BINS)  = (windows, TIME, FREQUENCY)
    #     sensor (n_windows, TS_SENSOR, 6)      — independent of Ta
    sensor_fingerprint = {}
    for split_label, samples in split_samples.items():
        x_a_all = np.stack([s["x_a"] for s in samples])
        tau_a_all = np.stack([s["tau_a"] for s in samples])
        x_s_all = np.stack([s["x_s"] for s in samples])
        tau_s_all = np.stack([s["tau_s"] for s in samples])
        assert x_a_all.shape == (len(samples), Ta, N_AUDIO_BINS), (
            f"Ta={Ta} FAILED: {split_label} audio array is {x_a_all.shape}, "
            f"expected (n_windows={len(samples)}, Ta={Ta}, N_AUDIO_BINS={N_AUDIO_BINS})")
        assert tau_a_all.shape == (len(samples), Ta), f"Ta={Ta} FAILED: {split_label} tau_a is {tau_a_all.shape}"
        assert x_s_all.shape == (len(samples), TS_SENSOR, 6), (
            f"Ta={Ta} FAILED: {split_label} sensor array is {x_s_all.shape}, expected (n_windows, {TS_SENSOR}, 6)")
        assert tau_s_all.shape == (len(samples), TS_SENSOR), f"Ta={Ta} FAILED: {split_label} tau_s is {tau_s_all.shape}"
        sensor_fingerprint[split_label] = (len(samples), round(float(x_s_all.astype(np.float64).sum()), 3),
                                           round(float(tau_s_all.astype(np.float64).sum()), 3),
                                           tuple(np.bincount([s["label"] for s in samples], minlength=5).tolist()))
        print(f"[SHAPE CHECK] {split_label:<5}: audio {x_a_all.shape} = (n_windows, Ta={Ta}, bins={N_AUDIO_BINS}) ✅ | "
              f"sensor {x_s_all.shape} = (n_windows, Ts={TS_SENSOR}, 6) ✅")
        del x_a_all, tau_a_all, x_s_all, tau_s_all

    cpw = np.array(_audio_chunks_per_window)
    n_short = int((cpw < Ta).sum())
    print(f"[AUDIO RESOLUTION] real audio chunks inside a {2.56:.2f}s window: min {cpw.min()} | median "
          f"{int(np.median(cpw))} | max {cpw.max()} -> Ta={Ta} "
          + ("sub-samples the real chunks in every window." if n_short == 0 else
             f"exceeds the available chunks in {n_short}/{len(cpw)} windows ({n_short / len(cpw) * 100:.1f}%), "
             "where some real chunk positions are repeated (no synthetic values are created)."))

    print("\n" + "=" * 60)
    print(f"     100% REAL PREPROCESSING COMPLETE (Ta={Ta}, ZERO SYNTHETIC DATA)   ")
    if _chunk_duration_log:
        _arr = np.array(_chunk_duration_log)
        print(f"Audio chunk duration check -> mean: {_arr.mean()*1000:.2f} ms | min: {_arr.min()*1000:.2f} ms | max: {_arr.max()*1000:.2f} ms")
    print("=" * 60)
    print(f"  • Total Train Diagnostic Windows : {len(split_samples['TRAIN'])}")
    print(f"  • Total Val Diagnostic Windows   : {len(split_samples['VAL'])}")
    print(f"  • Total Test Diagnostic Windows  : {len(split_samples['TEST'])}")
    print("=" * 60)

    out_dir = CACHE_DIR / f"Ta{Ta}"
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = {"train": out_dir / "bimodal_train.pt", "val": out_dir / "bimodal_val.pt", "test": out_dir / "bimodal_test.pt"}
    torch.save(split_samples["TRAIN"], paths["train"])
    torch.save(split_samples["VAL"], paths["val"])
    torch.save(split_samples["TEST"], paths["test"])
    print(f"✅ Cached real bimodal dataset for Ta={Ta} in {out_dir}")

    return {
        "paths": paths,
        "n_windows": (len(split_samples["TRAIN"]), len(split_samples["VAL"]), len(split_samples["TEST"])),
        "sensor_fingerprint": sensor_fingerprint,
        "chunks_per_window": (int(cpw.min()), int(np.median(cpw)), int(cpw.max())),
        "pct_windows_ta_exceeds_chunks": round(n_short / len(cpw) * 100, 2),
    }

print(f"\n✅ Preprocessing helpers ready — split + strongest-{N_AUDIO_BINS} bins fixed above; window "
      f"preprocessing (Ta = {TA_AUDIO}, Ts = {TS_SENSOR}) is built once in Cell 6 and reused for every beta.")

📥 Cloning official TII UAV Realistic Fault Dataset from GitHub...
Cloning into '/kaggle/working/tii_uav_dataset'...
remote: Enumerating objects: 324, done.
remote: Counting objects: 100% (324/324), done.
remote: Compressing objects: 100% (213/213), done.
remote: Total 324 (delta 111), reused 321 (delta 111), pack-reused 0 (from 0)
Receiving objects: 100% (324/324), 1.15 GiB | 25.80 MiB/s, done.
Resolving deltas: 100% (111/111), done.
Updating files: 100% (217/217), done.
✅ Dataset cloned successfully!
[NO-MODE-SWITCH CHECK] compute_audio_features('audio_chunks', 'bin_indices') | slice_real_bimodal has no window/compression/normalization parameters | source scan for hann/log1p/log_db/zscore/elif branches: clean ✅
[NO-MODE-SWITCH CHECK] no experiment globals (window / compression / normalization options) and no leftover N / Ta / sensor-feature globals defined ✅
Class 0 (19 flights) -> Train: 13, Val: 2, Test: 4
Class 1 (20 flights) -> Train: 14, Val: 3, Test: 3
Class 2 (20 flights) -> Tr

Bin selection N=64 (train-only audio scan):   0%|          | 0/69 [00:00<?, ?it/s]

[STRONGEST-64] Selected 64 bins from 356506 training-set audio chunks across 69 missions (FFT length = 769).
[STRONGEST-64] Selected bin indices (ascending): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760, 761, 762, 763, 764, 765, 766]
[STRONGEST-64] Estimated audio sample rate ≈ 84695.4 Hz (derived from audio buffer length and real inter-buffer timing across training missions; the dataset does not document a mic sample rate directly, so this is a data-derived estimate for labeling only, not a hardcoded spec).
[STRONGEST-64] Selected bins as 7 contiguous band(s) -> approx. frequency (Hz):
    bins   1- 11 ( 11 bins)  ->      55.1 -    606.5 Hz
    bins 245-255 ( 11 bins)  ->   13509.4 -  14060.8 Hz
    bins 258-267 ( 10 bins)  ->   14226.2 -  14722.4 Hz
    bins 501-

# Cell 5: DataLoader (weights_only=False for PyTorch 2.6+)

In [5]:
class RealBiModalDataset(Dataset):
    def __init__(self, pt_path: Path, mean=None, std=None):
        self.samples = torch.load(pt_path, weights_only=False)
        self.Ta = int(self.samples[0]["x_a"].shape[0])  # audio TIME positions per window
        self.Ts = 128

        if mean is None or std is None:
            all_s = np.concatenate([s["x_s"] for s in self.samples], axis=0)
            self.mean = all_s.mean(axis=0, keepdims=True)
            self.std = all_s.std(axis=0, keepdims=True) + 1e-6
        else:
            self.mean = mean
            self.std = std

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        norm_sensor = (s["x_s"] - self.mean) / self.std

        # --- FIX 1: use the REAL per-window timestamps saved during
        # preprocessing (Cell 4), normalized by that window's real duration,
        # instead of a synthetic torch.linspace(0.0, 1.0, ...) that was
        # identical for every sample regardless of the underlying data.
        window_sec = float(s["window_sec"])
        tau_a = torch.tensor(s["tau_a"], dtype=torch.float32) / window_sec
        tau_s = torch.tensor(s["tau_s"], dtype=torch.float32) / window_sec

        return {
            "x_a": torch.tensor(s["x_a"], dtype=torch.float32),
            "x_s": torch.tensor(norm_sensor, dtype=torch.float32),
            "tau_a": tau_a,
            "tau_s": tau_s,
            "label": torch.tensor(s["label"], dtype=torch.long),
        }

batch_size = 32

def build_dataloaders(paths, batch_size=batch_size):
    """Datasets + loaders (same settings as v7; identical for every beta)."""
    train_ds = RealBiModalDataset(paths["train"])
    val_ds   = RealBiModalDataset(paths["val"], mean=train_ds.mean, std=train_ds.std)
    test_ds  = RealBiModalDataset(paths["test"], mean=train_ds.mean, std=train_ds.std)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=2)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=2)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=2)

    print(f"DataLoaders ready -> Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)} windows")
    return train_ds, val_ds, test_ds, train_loader, val_loader, test_loader

print("RealBiModalDataset / build_dataloaders ready (DataLoaders are built per beta in Cell 6).")

RealBiModalDataset / build_dataloaders ready (DataLoaders are built per beta in Cell 6).


In [6]:
# ==============================================================================
# FIX 1 VERIFICATION HOOK — confirm tau_a / tau_s carry REAL, per-sample-varying
# timestamps, not the old synthetic torch.linspace(0.0, 1.0, ...) that was
# identical for every sample regardless of the underlying data.
# ==============================================================================
def verify_real_taus(train_ds):
    print("=" * 70)
    print("VERIFICATION HOOK — FIX 1: real per-sample tau_a / tau_s")
    print("=" * 70)

    check_idxs = [0, 1, len(train_ds) // 2, len(train_ds) - 1]
    taus_a, taus_s = [], []
    for idx in check_idxs:
        item = train_ds[idx]
        taus_a.append(item["tau_a"])
        taus_s.append(item["tau_s"])
        print(f"sample idx={idx:>6} | tau_a[:5]={item['tau_a'][:5].numpy()} | tau_s[:5]={item['tau_s'][:5].numpy()}")

    all_tau_a_identical = all(torch.allclose(taus_a[0], t) for t in taus_a[1:])
    all_tau_s_identical = all(torch.allclose(taus_s[0], t) for t in taus_s[1:])

    assert not all_tau_a_identical, (
        "❌ FIX 1 FAILED: tau_a is IDENTICAL across every sample checked — "
        "real timestamps are NOT reaching the model."
    )
    assert not all_tau_s_identical, (
        "❌ FIX 1 FAILED: tau_s is IDENTICAL across every sample checked — "
        "real timestamps are NOT reaching the model."
    )
    print("✅ FIX 1 VERIFIED: tau_a and tau_s vary across real samples (not synthetic/fixed).")
    print("=" * 70)

print("verify_real_taus(train_ds) ready — runs for every beta inside run_experiment.")

verify_real_taus(train_ds) ready — runs for every beta inside run_experiment.


# Cell 6: run_experiment(num_heads) — one isolated condition of the LACA attention-head-count sweep

In [7]:
# ==============================================================================
# run_experiment(num_heads) — ONE complete, isolated condition of the LACA
# attention-head-count sweep:
#   seeds reset -> v10 preprocessing (Ta=64, strongest-64, raw 6 IMU; built once,
#   identical for every head count) -> DataLoaders -> FRESH model (beta=1.0,
#   mlp g(Δτ), this condition's num_heads) + class-weighted loss + optimizer +
#   scheduler -> 20-epoch training with best-val checkpoint -> unseen-test
#   evaluation (+ learned lag Δ_as) -> GPU memory freed.
#
# How num_heads reaches the attention (Cell 2):
#   BiModalAeroSyncMamba(laca_num_heads=h) -> BiModalLACAFusion(num_heads=h)
#     -> laca_as / laca_sa: head_dim = d_model // h, scale = 1/sqrt(head_dim),
#        Q/K/V (each Linear(d_model, d_model)) viewed as (B, h, T, head_dim).
# The Q/K/V/out projections are d_model x d_model for EVERY h, so the head count
# only changes how the 256 dims are split (and the 1/sqrt(head_dim) scale) — the
# parameter count and initial weights should be identical across conditions;
# both are verified below rather than assumed.
# ==============================================================================
import gc
import hashlib

EPOCHS = 20
D_MODEL = 256
_PREP = None       # v10 preprocessing result (Ta=64), built once and reused by every head count
_INIT_REF = None   # first condition's init fingerprints, compared against every later head count


def head_count_problem(num_heads, d_model=D_MODEL):
    """None if num_heads gives an integer head_dim, else a human-readable reason to skip."""
    if not isinstance(num_heads, int) or num_heads < 1:
        return f"num_heads={num_heads!r} is not a positive integer"
    if d_model % num_heads != 0:
        return (f"d_model={d_model} is not divisible by num_heads={num_heads} "
                f"({d_model}/{num_heads} = {d_model / num_heads:.3f}, not an integer head_dim)")
    return None


def laca_param_counts(model):
    """LACA parameter counts read from the constructed model."""
    per_dir = {}
    for name in ("laca_as", "laca_sa"):
        laca = getattr(model.laca, name)
        per_dir[name] = {
            "qkv_out": sum(p.numel() for m in (laca.w_q, laca.w_k, laca.w_v, laca.out_proj) for p in m.parameters()),
            "lag_bias": sum(p.numel() for p in laca.lag_bias.parameters()),
            "total": sum(p.numel() for p in laca.parameters()),
        }
    total = sum(p.numel() for p in model.laca.parameters())   # both directions + gamma_as / gamma_sa
    return total, per_dir


def verify_head_config(model, num_heads):
    """Confirm, from the CONSTRUCTED LACA modules (not the config), which head split is in use."""
    head_dim = D_MODEL // num_heads
    for name in ("laca_as", "laca_sa"):
        laca = getattr(model.laca, name)
        print(f"[HEAD CHECK] model.laca.{name}: num_heads={laca.num_heads} | head_dim={laca.head_dim} | "
              f"d_model={laca.d_model} | scale={laca.scale:.6f} (1/sqrt({laca.head_dim}))")
        assert laca.num_heads == num_heads, f"{name}.num_heads = {laca.num_heads}, requested {num_heads}"
        assert laca.head_dim == head_dim and laca.num_heads * laca.head_dim == laca.d_model == D_MODEL, (
            f"{name}: num_heads*head_dim = {laca.num_heads}*{laca.head_dim} != d_model {laca.d_model}")
        assert abs(laca.scale - 1.0 / math.sqrt(head_dim)) < 1e-12, f"{name}.scale does not match head_dim"
        assert laca.beta == LACA_BETA, f"{name}.beta = {laca.beta}, expected the fixed {LACA_BETA}"
        g = laca.lag_bias
        assert (isinstance(g, nn.Sequential) and len(g) == 3 and isinstance(g[1], nn.Tanh)
                and (g[0].in_features, g[0].out_features, g[2].in_features, g[2].out_features) == (1, 32, 32, 1)), (
            f"{name}.lag_bias is not the fixed mlp g(Δτ): {g}")

    total, per_dir = laca_param_counts(model)
    for name, c in per_dir.items():
        print(f"[LACA PARAMS] {name}: Q/K/V/out {c['qkv_out']:,} + g(Δτ) mlp {c['lag_bias']:,} = {c['total']:,}")
    print(f"[LACA PARAMS] model.laca total (both directions + 2 gammas) = {total:,} | whole model = "
          f"{sum(p.numel() for p in model.parameters()):,}")
    print(f"[HEAD CHECK] ✅ both LACA directions use num_heads={num_heads}, head_dim={head_dim} "
          f"(beta={LACA_BETA}, g(Δτ)=mlp)")
    return total, per_dir


def init_fingerprints(model):
    """sha1 of the full initial state_dict and of the torch CPU RNG state right after
    construction. The head count changes no parameter shape, so both should be
    identical across all conditions."""
    h = hashlib.sha1()
    for k, v in model.state_dict().items():
        h.update(k.encode())
        h.update(v.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest(), hashlib.sha1(torch.get_rng_state().numpy().tobytes()).hexdigest()


def _manual_laca(laca, H_m, H_n, tau_m, tau_n, heads):
    """Independent re-implementation of PairwiseLACA.forward with an EXPLICIT head count."""
    B, T_m, D = H_m.shape
    T_n = H_n.shape[1]
    hd = D // heads
    Q = laca.w_q(H_m).view(B, T_m, heads, hd).transpose(1, 2)
    K = laca.w_k(H_n).view(B, T_n, heads, hd).transpose(1, 2)
    V = laca.w_v(H_n).view(B, T_n, heads, hd).transpose(1, 2)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(hd)
    delta_tau = (tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)
    B_mn = laca.lag_bias(delta_tau).squeeze(-1).unsqueeze(1)
    attn = F.softmax(scores + laca.beta * B_mn, dim=-1)
    out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T_m, D)
    return laca.out_proj(out), scores.shape


def verify_attention_shapes(model, batch_t, num_heads, when):
    """Run the model's OWN LACA modules on a real batch and assert every shape the
    downstream fusion step relies on; also check the output equals an independent
    computation with the requested head count (so the split is genuinely applied).
    Eval mode + no_grad: no RNG consumed, no BatchNorm stats updated."""
    xa, xs, ta, ts = batch_t
    was_training = model.training
    model.eval()
    with torch.no_grad():
        za = model.audio_encoder(xa)
        zs = model.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)
        Ha = model.htt(za, ta, modality_idx=0)
        Hs = model.htt(zs, ts, modality_idx=1)
        B, Ta_, Ts_ = Ha.shape[0], Ha.shape[1], Hs.shape[1]

        for name, laca, H_m, H_n, tau_m, tau_n in (("laca_as", model.laca.laca_as, Ha, Hs, ta, ts),
                                                   ("laca_sa", model.laca.laca_sa, Hs, Ha, ts, ta)):
            T_m, T_n = H_m.shape[1], H_n.shape[1]
            H_tilde, A_mn, lag = laca(H_m, H_n, tau_m, tau_n)
            assert H_tilde.shape == H_m.shape == (B, T_m, D_MODEL), f"{name}: output {tuple(H_tilde.shape)}"
            assert A_mn.shape == (B, T_m, T_n), f"{name}: head-averaged attention {tuple(A_mn.shape)}"
            assert lag.shape == (B,), f"{name}: expected lag {tuple(lag.shape)}"
            assert torch.isfinite(H_tilde).all() and torch.isfinite(A_mn).all(), f"{name}: non-finite output"
            assert torch.allclose(A_mn.sum(-1), torch.ones_like(A_mn.sum(-1)), atol=1e-4), f"{name}: rows != 1"

            manual, scores_shape = _manual_laca(laca, H_m, H_n, tau_m, tau_n, num_heads)
            assert scores_shape == (B, num_heads, T_m, T_n), f"{name}: scores {tuple(scores_shape)}"
            diff = float((H_tilde - manual).abs().max())
            assert diff < 1e-4, f"{name}: module output differs from a {num_heads}-head computation by {diff:.2e}"
            alt = 2 if num_heads == 1 else 1
            alt_diff = float((H_tilde - _manual_laca(laca, H_m, H_n, tau_m, tau_n, alt)[0]).abs().max())
            print(f"[ATTN {when}] {name}: scores {tuple(scores_shape)} -> output {tuple(H_tilde.shape)} ✅ | "
                  f"matches independent {num_heads}-head computation (max|diff| {diff:.1e}); "
                  f"a {alt}-head split would differ by {alt_diff:.1e}")

        # Downstream fusion step: residual gating, [H_a ; SEP ; H_s] concat, Mamba stack, classifier.
        H_tilde_a, H_tilde_s, aux = model.laca(Ha, Hs, ta, ts)
        assert H_tilde_a.shape == Ha.shape == (B, Ta_, D_MODEL) and H_tilde_s.shape == Hs.shape == (B, Ts_, D_MODEL)
        H_fused = torch.cat([H_tilde_a, model.sep_token.expand(B, -1, -1), H_tilde_s], dim=1)
        assert H_fused.shape == (B, Ta_ + 1 + Ts_, D_MODEL), f"fused sequence {tuple(H_fused.shape)}"
        logits, _, _, _ = model(xa, xs, ta, ts)
        assert logits.shape == (B, model.n_classes) and torch.isfinite(logits).all(), f"logits {tuple(logits.shape)}"
        print(f"[ATTN {when}] fusion: H̃_a {tuple(H_tilde_a.shape)} + SEP + H̃_s {tuple(H_tilde_s.shape)} -> "
              f"Mamba input {tuple(H_fused.shape)} -> logits {tuple(logits.shape)} ✅")
    model.train(was_training)


def run_experiment(num_heads, seed=SEED, epochs=EPOCHS):
    global _PREP, _INIT_REF
    problem = head_count_problem(num_heads)
    assert problem is None, f"invalid head count: {problem}"   # the driver skips these before calling
    head_dim = D_MODEL // num_heads
    t_run0 = time.time()
    set_all_seeds(seed)
    print(f"[SEEDS] reset at condition start -> random / numpy / torch / cuda = {seed}")
    print(f"[CONDITION] num_heads requested = {num_heads} -> head_dim = {D_MODEL} // {num_heads} = {head_dim} "
          f"(d_model {D_MODEL} divisible ✅) | beta={LACA_BETA}, g(Δτ)={LAG_BIAS_FORM} fixed")

    # --- 1. v10 preprocessing (Ta=64) — identical for every head count, built on first use ---
    if _PREP is None:
        _PREP = preprocess_for_ta(TA_AUDIO)
    else:
        print(f"[DATA] reusing the Ta={TA_AUDIO} windows built for the first head count (identical data for every condition)")
    prep = _PREP

    # --- 2. Datasets / loaders (v10 settings) ---
    train_ds, val_ds, test_ds, train_loader, val_loader, test_loader = build_dataloaders(prep["paths"])
    verify_real_taus(train_ds)

    # Batch built straight from the dataset (no DataLoader iteration -> no RNG consumed).
    probe = [train_ds[i] for i in range(min(batch_size, len(train_ds)))]
    probe_t = tuple(torch.stack([p[k] for p in probe]).to(device) for k in ("x_a", "x_s", "tau_a", "tau_s"))
    assert tuple(probe_t[0].shape) == (len(probe), TA_AUDIO, N_AUDIO_BINS)
    assert tuple(probe_t[1].shape) == (len(probe), TS_SENSOR, 6)

    # --- 3. FRESH model with this condition's num_heads (seed reset right before, exactly as v5-v10) ---
    set_all_seeds(seed)
    model = BiModalAeroSyncMamba(
        d_model=D_MODEL,
        n_classes=5,
        num_mamba_blocks=4,
        dropout=0.15,
        temporal_bias_scale=LACA_BETA,   # FIXED at 1.0 (v9 winner)
        audio_in_dim=N_AUDIO_BINS,
        sensor_in_dim=6,
        lag_bias_form=LAG_BIAS_FORM,     # FIXED "mlp" (v10 winner)
        laca_num_heads=num_heads,        # -> the ONLY thing that changes between conditions
    ).to(device)
    init_fp, rng_fp = init_fingerprints(model)

    print("\n" + "-" * 90)
    laca_total, laca_per_dir = verify_head_config(model, num_heads)
    if _INIT_REF is None:
        _INIT_REF = {"heads": num_heads, "init": init_fp, "rng": rng_fp}
        print(f"[INIT] reference fingerprints recorded from num_heads={num_heads}: initial weights {init_fp[:12]} | "
              f"RNG after construction {rng_fp[:12]}")
    else:
        same_init, same_rng = init_fp == _INIT_REF["init"], rng_fp == _INIT_REF["rng"]
        print(f"[INIT] ALL initial weights identical to the num_heads={_INIT_REF['heads']} condition: "
              f"{'YES ✅' if same_init else 'NO ❌'} | torch RNG state after construction identical "
              f"(-> same shuffle order + dropout masks): {'YES ✅' if same_rng else 'NO ❌'}")
        if not (same_init and same_rng):
            print("🚨" * 40 + "\n🚨  initial weights / RNG stream differ between head counts — this comparison "
                  "is NOT isolating num_heads alone.\n" + "🚨" * 40)
    print(f"[CONDITION] data: Ta={TA_AUDIO} x {N_AUDIO_BINS} bins audio, Ts={TS_SENSOR} x 6 raw IMU channels "
          f"(v10, unchanged)")
    verify_attention_shapes(model, probe_t, num_heads, "before training")
    print("-" * 90 + "\n")

    _fusion_cls = type(model.mamba_blocks[0])
    print(f"[MODEL] FUSION BACKBONE IN USE: {_fusion_cls.__module__}.{_fusion_cls.__name__}"
          + ("  🚨 pure-PyTorch FALLBACK, NOT real mamba_ssm 🚨" if _fusion_cls.__name__ == "VectorizedSSMBlock"
             else "  ✅ REAL mamba_ssm selective-scan SSM"))

    # FIX 4: inverse-frequency class weights from REAL training-set class counts
    train_labels_arr = np.array([s["label"] for s in train_ds.samples])
    class_counts = np.bincount(train_labels_arr, minlength=5)
    class_weights_np = class_counts.sum() / (len(class_counts) * np.maximum(class_counts, 1))
    class_weights = torch.tensor(class_weights_np, dtype=torch.float32).to(device)
    print("[FIX 4] class weights (from real train counts): "
          + " | ".join(f"C{c}: n={class_counts[c]} w={class_weights_np[c]:.4f}" for c in range(5)))

    criterion = BiModalAeroSyncLoss(lambda_align=0.1, lambda_cons=0.1, class_weights=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    best_val_f1 = 0.0
    best_weights = None
    best_epoch = None
    history = {"train_loss": [], "val_loss": [], "val_f1": [], "val_acc": []}
    ckpt_path = CACHE_DIR / f"aerosync_bimodal_best_heads{num_heads}.pt"
    tag = f"heads={num_heads}"

    try:
        # --- 4. Training (identical loop to v5-v10) ---
        start_time = time.time()
        print(f"🚀 Training Bi-Modal AeroSync-Mamba ({tag}) on {len(train_ds)} real flight windows...\n")

        for epoch in range(1, epochs + 1):
            model.train()
            running_loss, running_cls = 0.0, 0.0

            pbar = tqdm(train_loader, desc=f"{tag} Epoch {epoch:02d}/{epochs}", leave=False)
            for batch in pbar:
                xa = batch["x_a"].to(device, non_blocking=True)
                xs = batch["x_s"].to(device, non_blocking=True)
                ta = batch["tau_a"].to(device, non_blocking=True)
                ts = batch["tau_s"].to(device, non_blocking=True)
                labels = batch["label"].to(device, non_blocking=True)

                optimizer.zero_grad()
                logits, pooled_reps, aux_preds, _ = model(xa, xs, ta, ts)
                loss, l_dict = criterion(logits, labels, pooled_reps, aux_preds)
                loss.backward()

                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                running_loss += loss.item()
                running_cls += l_dict["loss_cls"]
                pbar.set_postfix({"loss": f"{loss.item():.3f}", "cls": f"{l_dict['loss_cls']:.3f}"})

            scheduler.step()
            avg_train_loss = running_loss / len(train_loader)

            # --- Validation ---
            model.eval()
            val_preds, val_targets, val_probs = [], [], []
            val_loss_total = 0.0

            with torch.no_grad():
                for batch in val_loader:
                    xa = batch["x_a"].to(device)
                    xs = batch["x_s"].to(device)
                    ta = batch["tau_a"].to(device)
                    ts = batch["tau_s"].to(device)
                    labels = batch["label"].to(device)

                    logits, pooled_reps, aux_preds, _ = model(xa, xs, ta, ts)
                    v_loss, _ = criterion(logits, labels, pooled_reps, aux_preds)
                    val_loss_total += v_loss.item()

                    probs = F.softmax(logits, dim=-1)
                    val_preds.extend(probs.argmax(dim=-1).cpu().numpy())
                    val_targets.extend(labels.cpu().numpy())
                    val_probs.extend(probs.cpu().numpy())

            avg_val_loss = val_loss_total / len(val_loader)
            val_metrics = compute_all_metrics(np.array(val_targets), np.array(val_preds), np.array(val_probs))

            history["train_loss"].append(avg_train_loss)
            history["val_loss"].append(avg_val_loss)
            history["val_f1"].append(val_metrics["macro_f1"])
            history["val_acc"].append(val_metrics["accuracy"])

            is_best = ""
            if val_metrics["macro_f1"] > best_val_f1:
                best_val_f1 = val_metrics["macro_f1"]
                best_weights = copy.deepcopy(model.state_dict())
                best_epoch = epoch
                torch.save(best_weights, ckpt_path)
                is_best = " ⭐ [Best Saved]"

            print(
                f"{tag} Epoch {epoch:02d}/{epochs} | "
                f"Train Loss: {avg_train_loss:.4f} | "
                f"Val Loss: {avg_val_loss:.4f} | "
                f"Val Acc: {val_metrics['accuracy']:.4f} | "
                f"Val Macro-F1: {val_metrics['macro_f1']:.4f}{is_best}"
            )

        train_seconds = time.time() - start_time
        print(f"\n🎉 {tag} Training Finished in {train_seconds:.1f}s | Best Val Macro-F1: {best_val_f1:.4f} "
              f"(epoch {best_epoch})")

        # --- 5. Unseen test evaluation with the best-val weights (as v5 Cell 7, incl. learned lag) ---
        if best_weights is not None:
            model.load_state_dict(best_weights)
        model.eval()
        verify_attention_shapes(model, probe_t, num_heads, "best checkpoint")

        test_preds, test_targets, test_probs, lags_as_list, lags_sa_list = [], [], [], [], []
        with torch.no_grad():
            for batch in test_loader:
                xa = batch["x_a"].to(device)
                xs = batch["x_s"].to(device)
                ta = batch["tau_a"].to(device)
                ts = batch["tau_s"].to(device)
                labels = batch["label"].to(device)

                logits, _, _, aux = model(xa, xs, ta, ts)
                probs = F.softmax(logits, dim=-1)
                test_preds.extend(probs.argmax(dim=-1).cpu().numpy())
                test_targets.extend(labels.cpu().numpy())
                test_probs.extend(probs.cpu().numpy())
                lags_as_list.extend(aux["lag_as"].cpu().numpy())
                lags_sa_list.extend(aux["lag_sa"].cpu().numpy())

        test_metrics = compute_all_metrics(np.array(test_targets), np.array(test_preds), np.array(test_probs))
        lag_as_mean = float(np.mean(lags_as_list))
        lag_as_std = float(np.std(lags_as_list))
        lag_sa_mean = float(np.mean(lags_sa_list))
        n_test = len(test_targets)
        print("=" * 65)
        print(f"     BI-MODAL AEROSYNC-MAMBA ({tag}, head_dim={head_dim}): 100% REAL TEST BENCHMARK")
        print("=" * 65)
        print(f"  • Test Accuracy       : {test_metrics['accuracy']:.4f} ({test_metrics['accuracy']*100:.2f}%)")
        print(f"  • Macro-F1 Score      : {test_metrics['macro_f1']:.4f}")
        print(f"  • Balanced Accuracy   : {test_metrics['balanced_accuracy']:.4f}")
        print(f"  • One-vs-Rest AUROC   : {test_metrics['macro_auroc']:.4f}")
        print(f"  • Learned Audio-IMU Lag (Δ_as) : {lag_as_mean:+.4f} (normalized units) "
              f"≈ {lag_as_mean * WINDOW_SEC * 1000:+.1f} ms | std over test windows {lag_as_std:.4f}")
        print(f"  • Learned IMU-Audio Lag (Δ_sa) : {lag_sa_mean:+.4f} (normalized units)")
        print(f"  • LACA parameters     : {laca_total:,} (both directions)")
        print("=" * 65)
    finally:
        # --- 6. Free GPU memory so sequential conditions never accumulate ---
        del model, optimizer, scheduler, criterion, best_weights, probe_t, probe
        del train_loader, val_loader, test_loader, train_ds, val_ds, test_ds
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            print(f"[GPU] freed | allocated={torch.cuda.memory_allocated() / 1024**2:.1f} MB "
                  f"| reserved={torch.cuda.memory_reserved() / 1024**2:.1f} MB")

    return {
        "num_heads": num_heads,
        "head_dim": head_dim,
        "accuracy": float(test_metrics["accuracy"]),
        "macro_f1": float(test_metrics["macro_f1"]),
        "balanced_accuracy": float(test_metrics["balanced_accuracy"]),
        "macro_auroc": float(test_metrics["macro_auroc"]),
        "learned_lag_as_mean": round(lag_as_mean, 6),
        "train_seconds": round(train_seconds, 1),
        "laca_params": laca_total,
        "laca_params_per_direction": laca_per_dir["laca_as"]["total"],
        "learned_lag_as_std": round(lag_as_std, 6),
        "learned_lag_as_ms": round(lag_as_mean * WINDOW_SEC * 1000, 2),
        "learned_lag_sa_mean": round(lag_sa_mean, 6),
        "best_val_macro_f1": round(float(best_val_f1), 6),
        "best_epoch": best_epoch,
        "n_test_windows": n_test,
        "seed": seed,
        "run_seconds": round(time.time() - t_run0, 1),
        # num_heads=4 reproduction fingerprints (not written to the CSV)
        "_n_windows": prep["n_windows"],
        "_class_counts": class_counts.tolist(),
        "_sensor_fingerprint": prep["sensor_fingerprint"],
        "_epoch1": {"train_loss": history["train_loss"][0], "val_loss": history["val_loss"][0],
                    "val_acc": history["val_acc"][0], "val_f1": history["val_f1"][0]},
    }

WINDOW_SEC = 2.56   # slice_real_bimodal window length; tau is normalized by it in RealBiModalDataset
print(f"run_experiment(num_heads) ready — sweep: num_heads = {NUM_HEADS_VALUES} | fixed: d_model={D_MODEL}, "
      f"beta={LACA_BETA}, g(Δτ)={LAG_BIAS_FORM}, Ta={TA_AUDIO}, N={N_AUDIO_BINS}, Ts={TS_SENSOR}, raw 6 IMU channels")

run_experiment(num_heads) ready — sweep: num_heads = [1, 2, 4, 8, 16] | fixed: d_model=256, beta=1.0, g(Δτ)=mlp, Ta=64, N=64, Ts=128, raw 6 IMU channels


# Cell 7: Sweep Driver — num_heads = 1, 2, 4, 8, 16 (beta=1.0, mlp g(Δτ) fixed), CSV row after each, num_heads=4 checked against v10

In [8]:
# ==============================================================================
# DRIVER — num_heads = 1, 2, 4, 8, 16 in the listed order, one notebook
# execution. Invalid head counts (d_model not divisible) are SKIPPED with a
# warning. Each row is appended to the CSV the moment its condition finishes
# (timeout-safe). The num_heads=4 reproduction check against v10 prints
# IMMEDIATELY after that condition.
# ==============================================================================
import csv
import traceback
from datetime import datetime

SWEEP_CSV = Path("/kaggle/working/laca_num_heads_sweep_results.csv")
SWEEP_CSV_FIELDS = ["num_heads", "head_dim", "accuracy", "macro_f1", "balanced_accuracy", "macro_auroc",
                    "learned_lag_as_mean", "train_seconds", "laca_params", "laca_params_per_direction",
                    "learned_lag_as_std", "learned_lag_as_ms", "learned_lag_sa_mean", "best_val_macro_f1",
                    "best_epoch", "n_test_windows", "seed", "run_seconds", "note", "finished_at"]

BASELINE_HEADS = 4
# If True, the sweep stops after a failed num_heads=4 reproduction instead of flagging it and continuing.
STOP_IF_BASELINE_MISMATCH = False

# Known result for this exact configuration (num_heads=4, mlp g(Δτ), beta=1.0, Ta=64, N=64, raw 6 IMU) — verified v5-v10.
V10_BASELINE_HEADS4 = {"accuracy": 0.9009, "macro_f1": 0.8966, "balanced_accuracy": 0.8940, "macro_auroc": 0.9884}
V5_LAG_AS_HEADS4 = -0.0196   # "Learned Audio-IMU Lag (Δ_as)" printed by v5 Cell 7 for this same model
# Deterministic fingerprints of the v7-v10 data path — independent of GPU numerics, must match EXACTLY.
V10_BIN_INDICES_N64 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255,
                       258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508,
                       509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760,
                       761, 762, 763, 764, 765, 766]
V10_N_WINDOWS = (4934, 954, 1130)
V10_CLASS_COUNTS = [906, 1020, 1004, 1008, 996]
V10_SENSOR_FINGERPRINT = {
    "TRAIN": (4934, -6167105.225, 808427.456, (906, 1020, 1004, 1008, 996)),
    "VAL":   (954, -1193206.652, 156301.664, (126, 229, 199, 190, 210)),
    "TEST":  (1130, -1413563.944, 185127.438, (286, 205, 210, 228, 201)),
}
V7_EPOCH1 = {"train_loss": 1.3863, "val_loss": 0.8762, "val_acc": 0.8249, "val_f1": 0.8390}
REPRO_TOL = 0.0005   # baseline numbers are reported to 4 decimals -> "near-identical" = within rounding

def check_heads4_against_v10(res):
    """Loud reproduction check for num_heads=4. Returns (ok, one-line verdict for the CSV)."""
    print("\n" + "=" * 90)
    print("  num_heads=4 SANITY CHECK — does the unchanged default reproduce the verified v10 baseline?")
    print("=" * 90)

    det = {
        "strongest-64 bin indices": SELECTED_BIN_INDICES.tolist() == V10_BIN_INDICES_N64,
        f"train/val/test windows {V10_N_WINDOWS}": tuple(res["_n_windows"]) == V10_N_WINDOWS,
        f"train class counts {V10_CLASS_COUNTS}": res["_class_counts"] == V10_CLASS_COUNTS,
        "raw sensor values/timestamps/labels (v10 fingerprint)": res["_sensor_fingerprint"] == V10_SENSOR_FINGERPRINT,
    }
    for name, ok in det.items():
        print(f"  [deterministic] {name:<53}: {'MATCH ✅' if ok else 'MISMATCH ❌'}")

    e1 = res["_epoch1"]
    print("  [epoch 1]       " + " | ".join(
        f"{k}: v7 {V7_EPOCH1[k]:.4f} vs now {e1[k]:.4f} ({e1[k] - V7_EPOCH1[k]:+.4f})" for k in V7_EPOCH1))

    print(f"\n  {'metric':<20} {'v10 known':>10} {'this run':>10} {'delta':>10}")
    worst = 0.0
    for k, v in V10_BASELINE_HEADS4.items():
        d = res[k] - v
        worst = max(worst, abs(d))
        print(f"  {k:<20} {v:>10.4f} {res[k]:>10.4f} {d:>+10.4f}{'' if abs(d) <= REPRO_TOL else '   <-- differs'}")
    print(f"  {'learned lag Δ_as':<20} {V5_LAG_AS_HEADS4:>+10.4f} {res['learned_lag_as_mean']:>+10.4f} "
          f"{res['learned_lag_as_mean'] - V5_LAG_AS_HEADS4:>+10.4f}   (v5 Cell 7 value; informational)")

    if not all(det.values()):
        ok, verdict = False, "PORT BUG: deterministic v10 fingerprint mismatch"
        print("\n" + "🚨" * 40)
        print("🚨  num_heads=4 DOES NOT REPRODUCE v10 — bins / windows / class counts / sensor values differ, which")
        print("🚨  cannot be GPU noise: the data path changed during the v10 -> v11 port. Do NOT trust the other")
        print("🚨  head counts until that is fixed.")
        print("🚨" * 40)
    elif worst <= REPRO_TOL:
        ok, verdict = True, "reproduces v10 (all 4 metrics within 0.0005)"
        print(f"\n✅ num_heads=4 REPRODUCES v10: all four test metrics within ±{REPRO_TOL} and all deterministic "
              f"fingerprints match.")
    else:
        ok, verdict = False, f"DOES NOT reproduce v10 metrics (max |delta| {worst:.4f}); data fingerprints match"
        print("\n" + "🚨" * 40)
        print(f"🚨  num_heads=4 TEST METRICS DIFFER FROM v10 (largest |delta| = {worst:.4f} > {REPRO_TOL}).")
        print("🚨  Data path is identical, so the difference arose in training. v5-v10 reproduced each other")
        print("🚨  exactly, so check the epoch-1 line: if epoch 1 already differs, the model/RNG path changed in")
        print("🚨  the port; if it matches and later epochs drift, it is GPU non-determinism. Treat every other")
        print("🚨  head count's difference with suspicion until this is explained.")
        print("🚨" * 40)
    print("=" * 90)
    return ok, verdict

def append_sweep_row(res, note=""):
    row = {k: res.get(k) for k in SWEEP_CSV_FIELDS}
    row["note"] = note
    row["finished_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    write_header = not SWEEP_CSV.exists()
    with open(SWEEP_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=SWEEP_CSV_FIELDS)
        if write_header:
            w.writeheader()
        w.writerow(row)
    print(f"💾 appended num_heads={res['num_heads']} to {SWEEP_CSV} ({'header + row' if write_header else 'row'})")

if SWEEP_CSV.exists():
    _backup = SWEEP_CSV.with_suffix(f".{datetime.now().strftime('%Y%m%d_%H%M%S')}.bak.csv")
    SWEEP_CSV.rename(_backup)
    print(f"ℹ️  existing results file moved to {_backup}")

# --- Divisibility pre-check for every candidate, before anything runs ---
print(f"HEAD-COUNT PRE-CHECK (d_model = {D_MODEL}):")
for h in NUM_HEADS_VALUES:
    p = head_count_problem(h)
    print(f"  num_heads={h!s:<3} -> " + (f"head_dim = {D_MODEL // h} ✅" if p is None else f"⚠️  WILL BE SKIPPED: {p}"))

sweep_failed, sweep_done, sweep_skipped = [], [], []
baseline_ok = None
_sweep_t0 = time.time()
for i, num_heads in enumerate(NUM_HEADS_VALUES, start=1):
    label = " — v10 BASELINE / sanity check" if num_heads == BASELINE_HEADS else ""
    print("\n\n" + "=" * 90)
    print(f"=== Running num_heads={num_heads} ({i}/{len(NUM_HEADS_VALUES)}){label} ===   "
          f"elapsed so far: {(time.time() - _sweep_t0) / 60:.1f} min")
    print("=" * 90)
    problem = head_count_problem(num_heads)
    if problem is not None:
        sweep_skipped.append((num_heads, problem))
        print(f"⚠️  SKIPPING num_heads={num_heads}: {problem}. PairwiseLACA reshapes Q/K/V (width d_model) into "
              f"num_heads x head_dim, which needs an exact split. Continuing with the remaining head counts.")
        continue
    try:
        res = run_experiment(num_heads)
    except Exception as exc:
        sweep_failed.append((num_heads, repr(exc)))
        print(f"\n❌ num_heads={num_heads} FAILED: {exc!r}")
        traceback.print_exc()
        if num_heads == BASELINE_HEADS:
            baseline_ok = False
            print("🚨" * 40 + "\n🚨  num_heads=4 failed — the v10 reproduction check could not run.\n" + "🚨" * 40)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if num_heads == BASELINE_HEADS and STOP_IF_BASELINE_MISMATCH:
            break
        continue
    note = ""
    if num_heads == BASELINE_HEADS:
        baseline_ok, note = check_heads4_against_v10(res)
    append_sweep_row(res, note)
    sweep_done.append(num_heads)
    if num_heads == BASELINE_HEADS and not baseline_ok and STOP_IF_BASELINE_MISMATCH:
        print("⛔ STOP_IF_BASELINE_MISMATCH=True -> stopping the sweep after the failed num_heads=4 sanity check.")
        break

print("\n\n" + "=" * 90)
print(f"SWEEP COMPLETE — {len(sweep_done)}/{len(NUM_HEADS_VALUES)} conditions finished in "
      f"{(time.time() - _sweep_t0) / 60:.1f} min | results: {SWEEP_CSV}")
for h, why in sweep_skipped:
    print(f"  ⚠️  num_heads={h}: skipped — {why}")
for h, err in sweep_failed:
    print(f"  ❌ num_heads={h}: {err}")
if baseline_ok is False:
    print("  🚨 the num_heads=4 sanity check did NOT pass — see the banner after that condition.")
print("=" * 90)

HEAD-COUNT PRE-CHECK (d_model = 256):
  num_heads=1   -> head_dim = 256 ✅
  num_heads=2   -> head_dim = 128 ✅
  num_heads=4   -> head_dim = 64 ✅
  num_heads=8   -> head_dim = 32 ✅
  num_heads=16  -> head_dim = 16 ✅


=== Running num_heads=1 (1/5) ===   elapsed so far: 0.0 min
[SEEDS] reset at condition start -> random / numpy / torch / cuda = 42
[CONDITION] num_heads requested = 1 -> head_dim = 256 // 1 = 256 (d_model 256 divisible ✅) | beta=1.0, g(Δτ)=mlp fixed

--- Preprocessing TRAIN split, Ta=64 (frozen strongest-64 bins) ---


Train Class 0 (Ta=64):   0%|          | 0/13 [00:00<?, ?it/s]

[STRONGEST-64 VERIFICATION] audio chunk shape (raw waveform): (5103, 1536) | np.fft.rfft output shape: (5103, 769) | final a_feats (sqrt-compressed, frozen strongest-64 bins) shape: (5103, 64)
[STRONGEST-64 VERIFICATION] Confirmed: features were produced via np.fft.rfft (frequency-domain magnitude spectrum) using the frozen strongest-64 bin indices, NOT the first 64 bins.


Train Class 1 (Ta=64):   0%|          | 0/14 [00:00<?, ?it/s]

Train Class 2 (Ta=64):   0%|          | 0/14 [00:00<?, ?it/s]

...

heads=16 Epoch 18/20 | Train Loss: 0.2517 | Val Loss: 0.7443 | Val Acc: 0.9046 | Val Macro-F1: 0.9084


heads=16 Epoch 19/20:   0%|          | 0/155 [00:00<?, ?it/s]

heads=16 Epoch 19/20 | Train Loss: 0.2310 | Val Loss: 0.7986 | Val Acc: 0.8994 | Val Macro-F1: 0.9041


heads=16 Epoch 20/20:   0%|          | 0/155 [00:00<?, ?it/s]

heads=16 Epoch 20/20 | Train Loss: 0.2410 | Val Loss: 0.7383 | Val Acc: 0.9078 | Val Macro-F1: 0.9114

🎉 heads=16 Training Finished in 245.1s | Best Val Macro-F1: 0.9181 (epoch 17)
[ATTN best checkpoint] laca_as: scores (32, 16, 64, 128) -> output (32, 64, 256) ✅ | matches independent 16-head computation (max|diff| 0.0e+00); a 1-head split would differ by 1.3e+00
[ATTN best checkpoint] laca_sa: scores (32, 16, 128, 64) -> output (32, 128, 256) ✅ | matches independent 16-head computation (max|diff| 0.0e+00); a 1-head split would differ by 4.7e+00
[ATTN best checkpoint] fusion: H̃_a (32, 64, 256) + SEP + H̃_s (32, 128, 256) -> Mamba input (32, 193, 256) -> logits (32, 5) ✅
     BI-MODAL AEROSYNC-MAMBA (heads=16, head_dim=16): 100% REAL TEST BENCHMARK
  • Test Accuracy       : 0.8947 (89.47%)
  • Macro-F1 Score      : 0.8906
  • Balanced Accuracy   : 0.8875
  • One-vs-Rest AUROC   : 0.9883
  • Learned Audio-IMU Lag (Δ_as) : -0.0146 (normalized units) ≈ -37.3 ms | std over test windows 0.0

# Cell 8: Final Comparison Table — sorted by macro-F1, LACA param count, num_heads=4 vs v10 baseline, differences vs noise floor

In [9]:
# ==============================================================================
# FINAL COMPARISON TABLE — reloaded from the CSV on disk (crash-safe source of
# truth), sorted by test macro-F1 descending, with the LACA parameter count,
# num_heads=4 vs the known v10 baseline, and every difference between head
# counts classified against the noise floor. Works on partial results too.
# ==============================================================================
NOISE_BAND_F1 = 0.0005   # spread between near-identical same-seed conditions in earlier sweeps

def noise_verdict(d_f1):
    if abs(d_f1) <= NOISE_BAND_F1:
        return "within noise floor — no real difference"
    return ("BETTER" if d_f1 > 0 else "WORSE") + " by more than the noise floor"

if not SWEEP_CSV.exists():
    raise FileNotFoundError(f"{SWEEP_CSV} does not exist — no condition finished. Re-run the driver cell.")

sweep_df = pd.read_csv(SWEEP_CSV).sort_values("macro_f1", ascending=False, kind="stable").reset_index(drop=True)
base_rows = sweep_df[sweep_df["num_heads"] == BASELINE_HEADS]
base = base_rows.iloc[0] if len(base_rows) else None
n_test = int(sweep_df.iloc[0]["n_test_windows"])

sweep_df.insert(0, "rank", np.arange(1, len(sweep_df) + 1))
if base is not None:
    sweep_df["Δ macro_f1 vs h=4"] = (sweep_df["macro_f1"] - base["macro_f1"]).round(4)
sweep_df["marker"] = [("🏆 top macro-F1" if i == 0 else "") + (" (v10 baseline / sanity check)" if h == BASELINE_HEADS else "")
                      for i, h in enumerate(sweep_df["num_heads"])]

cols = ["rank", "num_heads", "head_dim", "accuracy", "macro_f1", "balanced_accuracy", "macro_auroc",
        "learned_lag_as_mean", "learned_lag_as_ms", "laca_params", "train_seconds", "best_val_macro_f1", "best_epoch"] \
       + (["Δ macro_f1 vs h=4"] if base is not None else []) + ["marker"]
table = sweep_df[cols].copy()
for col in ("accuracy", "macro_f1", "balanced_accuracy", "macro_auroc", "learned_lag_as_mean"):
    table[col] = table[col].round(4)

n_valid = len([h for h in NUM_HEADS_VALUES if head_count_problem(h) is None])
pd.set_option("display.width", 260)
print("=" * 120)
print(f"   LACA NUM_HEADS SWEEP — num_heads ∈ {NUM_HEADS_VALUES}, d_model={D_MODEL}, beta={LACA_BETA}, "
      f"g(Δτ)={LAG_BIAS_FORM} fixed — sorted by test macro-F1 (descending)")
print(f"   ({len(sweep_df)}/{n_valid} valid conditions in the CSV) | laca_params = both LACA directions + 2 gammas")
print("=" * 120)
try:
    from IPython.display import display
    display(table)
except Exception:
    print(table.to_string(index=False))
print("=" * 120)

# --- 0. Skipped head counts (d_model not divisible) ---
for h in NUM_HEADS_VALUES:
    p = head_count_problem(h)
    if p is not None:
        print(f"⚠️  num_heads={h} was skipped: {p}")

# --- 1. LACA parameter count across head counts ---
counts = sorted(sweep_df["laca_params"].astype(int).unique())
if len(counts) == 1:
    print(f"LACA parameter count: {counts[0]:,} for EVERY head count — Q/K/V/out are {D_MODEL}x{D_MODEL} regardless "
          f"of num_heads; heads only change how those {D_MODEL} dims are split (and the 1/sqrt(head_dim) scale).")
else:
    print("LACA parameter count VARIES across head counts: "
          + ", ".join(f"h={int(r.num_heads)}: {int(r.laca_params):,}" for r in sweep_df.sort_values('num_heads').itertuples()))

# --- 2. num_heads=4 vs the known v10 baseline ---
if base is not None:
    print("\nnum_heads=4 (unchanged default) vs known v10 baseline:")
    for k, v in V10_BASELINE_HEADS4.items():
        d = base[k] - v
        print(f"  {k:<20} v10 {v:.4f} | this run {base[k]:.4f} | delta {d:+.4f}"
              + ("" if abs(d) <= REPRO_TOL else "   <-- differs"))
    print(f"  reproduction verdict: {base['note']}")
    if not str(base["note"]).startswith("reproduces"):
        print("🚨" * 40 + "\n🚨  the num_heads=4 sanity check did NOT pass — the head-count comparison below is not "
              "trustworthy until that is explained.\n" + "🚨" * 40)
else:
    print("\n🚨 num_heads=4 is not in the CSV — the v10 reproduction check did not complete, so there is no anchored baseline.")

# --- 3. Every head count vs num_heads=4, and every pair, against the noise floor ---
print(f"\nDIFFERENCES vs NOISE FLOOR (±{NOISE_BAND_F1} macro-F1; one test window = {1 / n_test:.4f} accuracy):")
if base is not None:
    for _, r in sweep_df.sort_values("num_heads").iterrows():
        if r["num_heads"] == BASELINE_HEADS:
            continue
        d = r["macro_f1"] - base["macro_f1"]
        d_win = int(round((r["accuracy"] - base["accuracy"]) * n_test))
        print(f"  num_heads={int(r['num_heads']):<3} (head_dim {int(r['head_dim']):>3}) vs 4 : Δ macro-F1 {d:+.4f} "
              f"({d_win:+d} test windows) | Δ bal-acc {r['balanced_accuracy'] - base['balanced_accuracy']:+.4f} | "
              f"Δ AUROC {r['macro_auroc'] - base['macro_auroc']:+.4f} -> {noise_verdict(d)}")
rows = list(sweep_df.itertuples(index=False))
beyond = []
for a in range(len(rows)):
    for b in range(a + 1, len(rows)):
        d = rows[a].macro_f1 - rows[b].macro_f1
        if abs(d) > NOISE_BAND_F1:
            beyond.append(f"h={int(rows[a].num_heads)} vs h={int(rows[b].num_heads)} ({d:+.4f})")
if len(rows) >= 2:
    spread = sweep_df["macro_f1"].max() - sweep_df["macro_f1"].min()
    print(f"  max macro-F1 spread across head counts: {spread:.4f}")
    if beyond:
        print(f"  ➜ YES — {len(beyond)} pairwise difference(s) LARGER than the ±{NOISE_BAND_F1} noise floor: "
              + "; ".join(beyond))
    else:
        print(f"  ➜ NO — every pairwise macro-F1 difference is within the ±{NOISE_BAND_F1} noise floor.")

top = sweep_df.iloc[0]
print()
if base is None:
    print(f"🏆 NOMINAL TOP: num_heads={int(top['num_heads'])} — macro-F1 {top['macro_f1']:.4f} (no num_heads=4 row to compare against)")
elif top["num_heads"] == BASELINE_HEADS:
    print("🏆 OVERALL: num_heads=4 (the current default) has the highest test macro-F1.")
elif top["macro_f1"] - base["macro_f1"] > NOISE_BAND_F1:
    print(f"🏆 OVERALL WINNER: num_heads={int(top['num_heads'])} — macro-F1 {top['macro_f1']:.4f}, "
          f"{top['macro_f1'] - base['macro_f1']:+.4f} vs num_heads=4, LARGER than the ±{NOISE_BAND_F1} noise floor.")
else:
    print(f"🏆 NOMINAL TOP: num_heads={int(top['num_heads'])} — macro-F1 {top['macro_f1']:.4f} "
          f"({top['macro_f1'] - base['macro_f1']:+.4f} vs num_heads=4), WITHIN the ±{NOISE_BAND_F1} noise floor: "
          f"no clear winner over the current default.")

# --- 4. Learned lag per head count ---
print("\nLEARNED LAG Δ_as PER HEAD COUNT (normalized units | ms):")
for _, r in sweep_df.sort_values("num_heads").iterrows():
    print(f"  num_heads={int(r['num_heads']):<3} Δ_as {r['learned_lag_as_mean']:+.4f} | {r['learned_lag_as_ms']:+7.1f} ms "
          f"| std over test windows {r['learned_lag_as_std']:.4f}")

print("\n   Caveat: same-seed reruns here are bit-exact, so the noise floor is the spread between near-identical "
      "conditions, not a measured seed-to-seed variance; one seed per condition cannot establish a small effect.")
if len(sweep_df) < n_valid:
    print(f"⚠️  only {len(sweep_df)}/{n_valid} valid conditions finished — the comparison covers those only.")
print(f"📄 full results CSV: {SWEEP_CSV}")

   LACA NUM_HEADS SWEEP — num_heads ∈ [1, 2, 4, 8, 16], d_model=256, beta=1.0, g(Δτ)=mlp fixed — sorted by test macro-F1 (descending)
   (5/5 valid conditions in the CSV) | laca_params = both LACA directions + 2 gammas


,rank,num_heads,head_dim,accuracy,macro_f1,balanced_accuracy,macro_auroc,learned_lag_as_mean,learned_lag_as_ms,laca_params,train_seconds,best_val_macro_f1,best_epoch,Δ macro_f1 vs h=4,marker
0,1,1,256,0.9018,0.8974,0.8945,0.9887,-0.0104,-26.58,526532,337.7,0.899257,20,0.0008,🏆 top macro-F1
1,2,4,64,0.9009,0.8966,0.8940,0.9884,-0.0196,-50.19,526532,313.0,0.908911,17,0.0000,(v10 baseline / sanity check)
2,3,16,16,0.8947,0.8906,0.8875,0.9883,-0.0146,-37.29,526532,245.1,0.918107,17,-0.0061,
3,4,8,32,0.8929,0.8889,0.8852,0.9875,-0.0164,-41.96,526532,238.1,0.904941,10,-0.0077,
4,5,2,128,0.8894,0.8848,0.8825,0.9874,-0.0241,-61.63,526532,231.8,0.903070,10,-0.0119,


LACA parameter count: 526,532 for EVERY head count — Q/K/V/out are 256x256 regardless of num_heads; heads only change how those 256 dims are split (and the 1/sqrt(head_dim) scale).

num_heads=4 (unchanged default) vs known v10 baseline:
  accuracy             v10 0.9009 | this run 0.9009 | delta -0.0000
  macro_f1             v10 0.8966 | this run 0.8966 | delta +0.0000
  balanced_accuracy    v10 0.8940 | this run 0.8940 | delta -0.0000
  macro_auroc          v10 0.9884 | this run 0.9884 | delta -0.0000
  reproduction verdict: reproduces v10 (all 4 metrics within 0.0005)

DIFFERENCES vs NOISE FLOOR (±0.0005 macro-F1; one test window = 0.0009 accuracy):
  num_heads=1   (head_dim 256) vs 4 : Δ macro-F1 +0.0008 (+1 test windows) | Δ bal-acc +0.0005 | Δ AUROC +0.0003 -> BETTER by more than the noise floor
  num_heads=2   (head_dim 128) vs 4 : Δ macro-F1 -0.0119 (-13 test windows) | Δ bal-acc -0.0115 | Δ AUROC -0.0010 -> WORSE by more than the noise floor
  num_heads=8   (head_dim  32) vs 4